In [ ]:
# -*- coding: utf-8 -*-


# AI-Based Financial Advisory Model — Research Prototype

# 0.0 PROTOTYPE SETUP

### 0.1. Library Installation

First, we install all the necessary libraries that will be used across the different modules of the financial advisory research prototype. This ensures all dependencies are met before proceeding with data loading and processing.


In [ ]:
import sys

# Install required libraries
!{sys.executable} -m pip install pandas numpy scikit-learn scikit-fuzzy prophet transformers torch matplotlib seaborn

# Install evaluation libraries if needed (e.g., from a specific version or for specific models)
# !{sys.executable} -m pip install evaluate

print("Libraries installed successfully.")


This code cell installs all necessary Python libraries using `pip`. It includes `pandas`, `numpy`, `scikit-learn`, `scikit-fuzzy`, `prophet`, `transformers`, `torch`, `matplotlib`, and `seaborn` to ensure all dependencies are met for the financial advisory prototype.

### 0.2. Library Imports

Next, we import the installed libraries. This step makes the functionalities of these libraries available for use in the notebook. We'll import them now to verify they are all accessible.


In [ ]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from prophet import Prophet
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns


This code cell imports the installed libraries, making their functionalities available for use throughout the notebook. Key libraries imported include `pandas` for data manipulation, `numpy` for numerical operations, `skfuzzy` for fuzzy logic, `Prophet` for time series forecasting, `transformers` for NLP, and `matplotlib` and `seaborn` for visualization.

### 0.3. Configuration Variables

We define essential configuration variables, including a random seed for reproducibility. It is crucial to define these upfront to maintain consistency and facilitate reproducibility of the research.


In [ ]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Output directory
OUTPUT_DIR = 'output_results'

# Target monthly gross income range
TARGET_MIN_INCOME = 24060.0
TARGET_MAX_INCOME = 144360.0

# Prophet forecast configuration
FORECAST_HORIZON_DAYS = 30

# Market signal thresholds
POSITIVE_THRESHOLD = 0.01
NEGATIVE_THRESHOLD = -0.01
FORECAST_UNCERTAINTY_THRESHOLD = 0.06

print(f"Random Seed: {RANDOM_SEED}")
print(f"Output Directory: {OUTPUT_DIR}")
print(
    f"Target Income Range: "
    f"₱{TARGET_MIN_INCOME:,.0f} - "
    f"₱{TARGET_MAX_INCOME:,.0f}"
)
print(f"Forecast Horizon: {FORECAST_HORIZON_DAYS} business days")


This code cell defines essential configuration variables for the project. It sets a `RANDOM_SEED` for reproducibility, specifies an `OUTPUT_DIR` for saving results, defines `TARGET_MIN_INCOME` and `TARGET_MAX_INCOME` for income screening, sets the `FORECAST_HORIZON_DAYS` for Prophet, and establishes `POSITIVE_THRESHOLD`, `NEGATIVE_THRESHOLD`, and `FORECAST_UNCERTAINTY_THRESHOLD` for market signal classification.


In [ ]:
import os

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print(
    f"Paper-ready outputs will be saved to: "
    f"{OUTPUT_DIR}"
)


This code cell creates the specified output directory if it doesn't already exist. This ensures that all generated figures and data tables can be saved to a designated location without errors.

All specified libraries have been installed and imported, and the initial configuration variables are defined. We are now ready to proceed with data loading.

# 1.0. LOAD DATASET

### 1.1. Data Loading

We'll use Google Colab's file upload feature to load the survey dataset. Once uploaded, the data will be loaded into a pandas DataFrame. We'll then display initial insights into the dataset, including the number of rows and columns, column names, the first five rows, and data types, to ensure proper loading and to facilitate early inspection.


In [ ]:
import io
import re
from google.colab import files

# Upload the CSV file
uploaded = files.upload()

# Assuming only one file is uploaded, get its name
file_name = next(iter(uploaded))

# Read uploaded survey dataset
df_original = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\nSuccessfully loaded '{file_name}'.")
print(f"Initial rows: {df_original.shape[0]}")

# Keep original dataset untouched
income_screening_col = (
    'Do you have a monthly gross income range between ₱24,060–₱144,360?'
)

if income_screening_col not in df_original.columns:
    raise ValueError(
        f"Income screening column not found: {income_screening_col}"
    )

# Create filtered working copy
df_working = df_original[
    df_original[income_screening_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('yes')
].copy()

print(
    f"Rows after income-screening filter: "
    f"{len(df_working)}"
)

# Create stable respondent IDs AFTER filtering
RESPONDENT_ID_COLUMN = 'Respondent_ID'

df_working.insert(
    0,
    RESPONDENT_ID_COLUMN,
    np.arange(1, len(df_working) + 1)
)

print(
    f"Created '{RESPONDENT_ID_COLUMN}' "
    "without modifying df_original."
)

display(
    df_working[
        [RESPONDENT_ID_COLUMN, 'Timestamp', 'Age']
    ].head()
)


This code cell handles the upload of the survey dataset from Google Colab's file upload feature. It reads the CSV into a pandas DataFrame, `df_original`, and then creates a filtered working copy, `df_working`, by screening respondents based on their monthly gross income. It also assigns a unique `Respondent_ID` to each eligible respondent.


In [ ]:
df_working.to_csv('filtered_respondents.csv', index=False)

files.download('filtered_respondents.csv')


This code cell saves the filtered `df_working` DataFrame to a CSV file named `filtered_respondents.csv` and then provides a download link for the user to retrieve this file. This step ensures that the pre-processed data can be easily accessed or stored externally.

### 1.2. Identify Respondent ID and Create Working Copy

df_original remains the untouched uploaded survey, while df_working contains only eligible respondents and all subsequent preprocessing operations.

# 2.0. DATA VALIDATION & PREPROCESSING

This section focuses on preparing the data for analysis. We'll start by categorizing the survey variables based on their intended use in the different analytical modules (Fuzzy C-Means for financial capacity, XLM-RoBERTa for behavioral analysis, and others for general purposes). After categorization, we'll perform essential data cleaning steps, including inspecting for missing values, duplicates, and converting data types. We will also perform basic text cleaning for the open-ended responses relevant to the XLM-RoBERTa module.

### 2.1. Variable Categorization

We will categorize the survey columns into three groups:

*   **FCM (Financial Capacity Module) Variables:** Structured/numerical variables used for financial capacity analysis.
*   **XLM-R (Behavioral Analysis Module) Variables:** Open-ended text variables for behavioral analysis.
*   **Other Survey Variables:** Remaining variables retained for later recommendation logic or analysis.


In [ ]:
# Initialize lists for categorized columns
fcm_cols = []
xlmr_cols = []
other_cols = []

# Exclude the RESPONDENT_ID_COLUMN from direct categorization, as it will be included in all datasets
feature_columns = [col for col in df_working.columns if col != RESPONDENT_ID_COLUMN]

# Manually categorize based on column names and expected data types
# FCM Variables (Structured/Numerical)
fcm_cols.extend([
    'Number of Dependents',
    '1. Average Monthly Gross Income',
    '2. Average Monthly Expenses',
    '3. Average Monthly Net Savings'
])

# XLM-R Variables (Open-ended Text)
xlmr_cols.extend([
    '7. How do you feel about investing in high-risk, high-reward platforms (like Crypto or Stocks) compared to keeping your money in a traditional bank? (Do you strictly prefer one over the other, or do you prefer to put half in the bank and half in investments?)',
    "9. Imagine you invested ₱10,000, and after one week, its value dropped to ₱7,000. What would you do and how would you feel? (e.g., Would you panic and sell everything, buy more because it's cheap, or just observe and wait?)"
])

# Other Survey Variables (Remaining columns)
# Automatically assign remaining columns to 'other_cols'
assigned_cols = set(fcm_cols + xlmr_cols + [RESPONDENT_ID_COLUMN])
other_cols = [col for col in df_working.columns if col not in assigned_cols]

print("--- Categorized Columns ---")
print(f"FCM (Financial Capacity) Columns: {fcm_cols}")
print(f"XLM-R (Behavioral Analysis) Columns: {xlmr_cols}")
print(f"Other Survey Columns: {other_cols}")

# Verify that all columns are covered
all_categorized = set(fcm_cols + xlmr_cols + other_cols + [RESPONDENT_ID_COLUMN])
if len(all_categorized) == len(df_working.columns):
    print("\nAll columns have been categorized.")
else:
    print("\nWARNING: Some columns were not categorized. Please review the lists.")
    missing_from_categorization = set(df_working.columns) - all_categorized
    print(f"Missing columns: {missing_from_categorization}")


This code cell categorizes the columns of the `df_working` DataFrame into three distinct groups: `fcm_cols` for the Fuzzy C-Means module, `xlmr_cols` for the XLM-RoBERTa module, and `other_cols` for remaining survey variables. This organization ensures that specific features are used by the appropriate analytical modules and verifies that all columns are covered.

### 2.2. Data Cleaning and Validation

This step involves inspecting the `df_working` DataFrame for common data quality issues and performing necessary conversions:

1.  **Missing Value Inspection:** Identify columns with missing data.
2.  **Duplicate Inspection:** Check for any fully duplicated rows.
3.  **Data Type Conversion:** Convert columns to appropriate numerical types, especially for FCM variables. Non-numeric values in these columns will be coerced to `NaN`.
4.  **Text Cleaning:** Apply basic cleaning to open-ended text responses for the XLM-R module.


In [ ]:
import re
import numpy as np
import pandas as pd

def clean_currency(val, no_means_zero=False):
    """
    Convert common Philippine financial responses into numeric values.

    Missing/unknown responses remain NaN.
    Explicit 'No' may be treated as zero only where logically valid,
    such as the debt question.
    """
    if pd.isna(val):
        return np.nan

    text = str(val).strip().lower()

    if text in ['', 'n/a', 'na', 'none', 'done']:
        return np.nan

    if text == 'no':
        return 0.0 if no_means_zero else np.nan

    text = (
        text.replace(',', '')
            .replace('php', '')
            .replace('₱', '')
            .replace('p', '')
            .replace(' ', '')
    )

    matches = re.findall(r'([\d\.]+)([km]?)', text)

    if not matches:
        return np.nan

    values = []
    global_mult = 1

    if matches[-1][1] == 'k':
        global_mult = 1000
    elif matches[-1][1] == 'm':
        global_mult = 1_000_000

    for num_str, suffix in matches:
        num = float(num_str)

        if suffix == 'k':
            num *= 1000
        elif suffix == 'm':
            num *= 1_000_000
        elif global_mult > 1 and num < 1000:
            num *= global_mult

        values.append(num)

    return max(values)


def clean_age(val):
    """
    Extract plain whole-number values for Age.
    Missing or illogical responses (like 'none') remain NaN.
    """
    if pd.isna(val):
        return np.nan

    text = str(val).strip().lower()

    if text in ['', 'none', 'n/a', 'na']:
        return np.nan

    # Handles "25 years old" -> 25
    match = re.search(r'(\d+)', text)
    if match:
        return int(match.group(1))

    return np.nan


def clean_dependents(val):
    """
    Extract plain whole-number values for Number of Dependents.
    Critically ensures that 'none' or 'zero' are treated as a legitimate 0.
    """
    if pd.isna(val):
        return 0

    text = str(val).strip().lower()

    # For this survey, a blank Number of Dependents is interpreted as having no dependents.
    if text in ['', 'n/a', 'na']:
        return 0

    # Explicitly catch strings that mean 0 dependents
    if text in ['none', 'none.', 'zero', 'no one', 'no']:
        return 0

    number_words = {
        'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
        'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
    }

    if text in number_words:
        return number_words[text]

    # Handles "0", "2", "3 dependents", etc.
    match = re.search(r'(\d+)', text)
    if match:
        return int(match.group(1))

    return np.nan


# STORE ORIGINAL UNCLEANED VALUES BEFORE PROCESSING
raw_income = df_working['1. Average Monthly Gross Income'].copy()

print("\n--- Applying Custom RegEx Cleaning for Messy User Data ---")

# Apply cleaning to Currency Columns
currency_columns = [
    '1. Average Monthly Gross Income',
    '2. Average Monthly Expenses',
    '3. Average Monthly Net Savings',
    '4. Total Average Educational Expenses (for further studies or of dependents)',
    '5. Do you have current loans/debts? (If yes, please specify the estimated total amount)'
]

debt_col = (
    '5. Do you have current loans/debts? '
    '(If yes, please specify the estimated total amount)'
)

for col in currency_columns:
    if col == debt_col:
        df_working[col] = df_working[col].apply(
            lambda x: clean_currency(x, no_means_zero=True)
        )
    else:
        df_working[col] = df_working[col].apply(clean_currency)

# Apply separated cleaning functions for Age and Dependents
df_working['Age'] = df_working['Age'].apply(clean_age)
print("Cleaned numeric column: 'Age'")

df_working['Number of Dependents'] = df_working['Number of Dependents'].apply(clean_dependents)
print("Cleaned numeric column: 'Number of Dependents'")


# --- INCOME BRACKET FILTER ---
income_value_col = '1. Average Monthly Gross Income'

print(
    f"\n--- Verifying Income Bracket "
    f"(₱{TARGET_MIN_INCOME:,.0f} - ₱{TARGET_MAX_INCOME:,.0f}) ---"
)

initial_count = len(df_working)

dropped_mask = (
    df_working[income_value_col].isna()
    |
    (df_working[income_value_col] < TARGET_MIN_INCOME)
    |
    (df_working[income_value_col] > TARGET_MAX_INCOME)
)

# Output cells being dropped
dropped_df = pd.DataFrame({
    'Respondent_ID':
        df_working.loc[dropped_mask, RESPONDENT_ID_COLUMN],

    'Original Input':
        raw_income.loc[dropped_mask],

    'Cleaned Value':
        df_working.loc[dropped_mask, income_value_col]
})

print(f"\nFlagged {len(dropped_df)} respondents for dropping:")
display(dropped_df)

df_working = df_working.loc[~dropped_mask].copy()

dropped_count = initial_count - len(df_working)

print(
    f"\nDropped {dropped_count} respondents whose income "
    "was missing or outside the target range."
)

print(
    f"Valid respondents remaining for AI pipeline: "
    f"{len(df_working)}"
)


This code cell defines custom functions (`clean_currency`, `clean_age`, `clean_dependents`) to preprocess messy user input in the DataFrame. It then applies these cleaning functions to relevant columns, converts the cleaned data to appropriate numerical types, and filters out respondents whose income values are still missing or fall outside the target range after cleaning.


In [ ]:
print("\n--- Verifying Data Type Conversion for FCM Variables ---")

# Define columns that should be numerical for FCM
numerical_fcm_columns = [
    '1. Average Monthly Gross Income',
    '2. Average Monthly Expenses',
    '3. Average Monthly Net Savings',
    '4. Total Average Educational Expenses (for further studies or of dependents)',
    '5. Do you have current loans/debts? (If yes, please specify the estimated total amount)'
]

# Enforce float64 type (safe to do AFTER the custom clean_currency function)
for col in numerical_fcm_columns:
    if col in df_working.columns:
        df_working[col] = df_working[col].astype('float64')
        print(f"Verified '{col}' as float64.")

# Enforce Int64 (nullable integer) for Age and Dependents
for col in ['Age', 'Number of Dependents']:
    if col in df_working.columns:
        df_working[col] = df_working[col].astype('Int64')
        print(f"Verified '{col}' as Int64.")

print("\n--- Current Data Types after conversion ---")
# Verify data types of the actual FCM variables
df_working[fcm_cols].info()


This code cell verifies and enforces the correct data types for the FCM (Financial Capacity Module) variables. It ensures that monetary columns are `float64` and 'Age' and 'Number of Dependents' are `Int64` (nullable integer), which is crucial for subsequent numerical analysis in the FCM module.

#### Categorical Encoding Scheme (DEFUNCT)

We will apply categorical encoding to several variables as specified:

*   **Ordinal Encoding for 5-point Behavioral Frequency Items:** For questions like "How often do you set aside money for savings?", "How often do you actively look for investment opportunities?", and "How often do you still have money left at the end of the month?", we will map the responses to a numerical scale:
    *   'Always': 5
    *   'Often': 4
    *   'Sometimes': 3
    *   'Rarely': 2
    *   'Never': 1

*   **One-Hot Encoding for Nominal Categorical Variables:** For 'Employment Status', 'Civil Status', and 'Educational Attainment', we will use one-hot encoding to convert them into numerical format without implying any order.

### 2.3. Text Cleaning for XLM-R Variables

We apply basic text cleaning to the open-ended responses to prepare them for the XLM-RoBERTa model. This typically involves operations like converting text to lowercase, removing extra whitespace, and potentially handling special characters.


In [ ]:
print("--- Text Cleaning for XLM-R Columns ---")

def clean_text(text):
    if pd.isna(text):
        return text
    text = str(text).lower()  # Convert to lowercase
    text = ' '.join(text.split())  # Remove extra whitespace
    # Further cleaning can be added here, e.g., removing punctuation or numbers
    return text

for col in xlmr_cols:
    if col in df_working.columns:
        df_working[col] = df_working[col].apply(clean_text)
        print(f"Cleaned text in '{col}'.")

print("\nFirst five rows of cleaned XLM-R text data:")
display(df_working[xlmr_cols].head())


This code cell applies basic text cleaning to the open-ended responses designated for the XLM-RoBERTa module. The `clean_text` function converts text to lowercase and removes extra whitespace, preparing the textual data for natural language processing by the model.

### 2.4. Create Separate DataFrames

Now, we create three distinct DataFrames, each tailored for a specific module, ensuring that the `RESPONDENT_ID_COLUMN` is included in all of them for traceability.

*   `df_fcm`: Contains the `RESPONDENT_ID_COLUMN` and the structured/numerical variables for the Fuzzy C-Means module.
*   `df_xlmr`: Contains the `RESPONDENT_ID_COLUMN` and the cleaned open-ended text variables for the XLM-RoBERTa module.
*   `df_other`: Contains the `RESPONDENT_ID_COLUMN` and any other variables not used in FCM or XLM-R, for future use or recommendation logic.


In [ ]:
# Create df_fcm
df_fcm = df_working[[RESPONDENT_ID_COLUMN] + fcm_cols].copy()
print("\n--- df_fcm (Financial Capacity Module) ---")
display(df_fcm.head())
display(df_fcm.info())

# Create df_xlmr
df_xlmr = df_working[[RESPONDENT_ID_COLUMN] + xlmr_cols].copy()
print("\n--- df_xlmr (Behavioral Analysis Module) ---")
display(df_xlmr.head())
display(df_xlmr.info())

# Dynamically figure out which columns are left over for df_other
used_cols = set(fcm_cols + xlmr_cols + [RESPONDENT_ID_COLUMN])
actual_remaining_cols = [col for col in df_working.columns if col not in used_cols]

# Create df_other
df_other = df_working[[RESPONDENT_ID_COLUMN] + actual_remaining_cols].copy()
print("\n--- df_other (Other Survey Variables) ---")
display(df_other.head())
display(df_other.info())


This code cell creates three separate DataFrames: `df_fcm` for the Fuzzy C-Means module, `df_xlmr` for the XLM-RoBERTa module, and `df_other` for all other survey variables. Each DataFrame includes the `RESPONDENT_ID_COLUMN` for traceability, and their initial rows and data types are displayed for verification.

### 2.5. Concise Validation Report




This report summarizes the findings from the data validation and preprocessing steps.


In [ ]:
print("--- Data Validation Report ---")
print(f"Original DataFrame Shape: {df_original.shape}")
print(f"Working DataFrame Shape: {df_working.shape}")

print(f"\nRespondent ID Column: '{RESPONDENT_ID_COLUMN}'")

# Missing values summary
missing_after_processing = df_working.isnull().sum()
missing_after_processing = missing_after_processing[missing_after_processing > 0]
if not missing_after_processing.empty:
    print("\nColumns with missing values after preprocessing:")
    display(missing_after_processing)
else:
    print("\nNo missing values found after preprocessing.")

# Calculate duplicates dynamically right here
print(f"\nNumber of duplicate rows found: {df_working.duplicated().sum()}")

print("\nDataFrames created for each module:")
print(f"  - df_fcm: Shape {df_fcm.shape}, Columns: {df_fcm.columns.tolist()}")
print(f"  - df_xlmr: Shape {df_xlmr.shape}, Columns: {df_xlmr.columns.tolist()}")
print(f"  - df_other: Shape {df_other.shape}, Columns: {df_other.columns.tolist()}")

print("\nPreprocessing complete. Ready for FCM Module.")


This code cell generates a concise data validation report. It summarizes the shapes of the original and working DataFrames, lists columns with missing values after preprocessing, checks for duplicate rows, and confirms the creation and dimensions of the `df_fcm`, `df_xlmr`, and `df_other` DataFrames, indicating readiness for the next stage.

### 2.6. Dataset Summary

This subsection summarizes the number of respondents retained at each preprocessing stage and presents descriptive statistics for the core financial variables used by the Financial Capacity Module. These tables are intended for reporting in the Results and Discussion chapter and do not modify the analytical dataset.


In [ ]:
print(
    "\n=== DATASET SUMMARY ==="
)

# FLOW

self_reported_income_eligible = (
    df_original[
        income_screening_col
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('yes')
    .sum()
)


missing_core_fcm = (
    df_working[
        fcm_cols
    ]
    .isna()
    .any(axis=1)
    .sum()
)


paper_sample_flow = pd.DataFrame({

    'Stage': [
        'Original survey responses',
        'Passed self-reported income screening',
        'Passed validated numeric income screening',
        'Incomplete core FCM records'
    ],

    'Respondents': [
        len(df_original),
        int(self_reported_income_eligible),
        len(df_working),
        int(missing_core_fcm)
    ]
})


print(
    "\n--- RESPONDENT FLOW ---"
)

display(
    paper_sample_flow
)


# DESCRIPTIVE FINANCIAL STATISTICS

descriptive_columns = [
    'Age',
    'Number of Dependents',
    '1. Average Monthly Gross Income',
    '2. Average Monthly Expenses',
    '3. Average Monthly Net Savings'
]


paper_descriptive_statistics = (
    df_working[
        descriptive_columns
    ]
    .describe()
    .T[
        [
            'count',
            'mean',
            'std',
            'min',
            '25%',
            '50%',
            '75%',
            'max'
        ]
    ]
    .round(2)
)


print(
    "\n--- DESCRIPTIVE STATISTICS ---"
)

display(
    paper_descriptive_statistics
)


# SAVE TABLES

paper_sample_flow.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_sample_flow.csv'
    ),
    index=False
)

paper_descriptive_statistics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_descriptive_statistics.csv'
    )
)

print(
    "\nDataset summary tables saved."
)


This code cell summarizes the dataset by presenting the respondent flow through various preprocessing stages (original, income-screened, validated income, and incomplete FCM records). It also provides descriptive statistics for key financial variables (Age, Dependents, Income, Expenses, Savings) in the `df_working` DataFrame and saves these summary tables to CSV files.

# 3.0 CORE ANALYTICAL MODULES

## 3.1. FINANCIAL CAPACITY MODULE: FUZZY C-MEANS

### 3.1.1. Data Preparation for Fuzzy C-Means

The FCM module uses Number of Dependents, Gross Income, Monthly Expenses, and Net Savings. Respondents with missing values in any of these core variables are excluded from the FCM branch. Monetary variables undergo Winsorization, logarithmic transformation, and standardization prior to clustering.


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Define the core variables used for FCM clustering.
# Debt and Educational Expenses are excluded from FCM because of high number of missing responses
fcm_features = [
    'Number of Dependents',
    '1. Average Monthly Gross Income',
    '2. Average Monthly Expenses',
    '3. Average Monthly Net Savings'
]

# Keep only Respondent ID + the selected FCM features
fcm_data = df_fcm[
    [RESPONDENT_ID_COLUMN] + fcm_features
].copy()

# Preserve an untransformed copy for later cluster interpretation
fcm_interpretation_data = fcm_data.copy()

# Identify respondents with at least one missing core FCM value
incomplete_fcm_mask = fcm_data[fcm_features].isna().any(axis=1)

num_incomplete = incomplete_fcm_mask.sum()

print(f"Total respondents available before FCM filtering: {len(fcm_data)}")
print(f"Respondents excluded from FCM due to incomplete core data: {num_incomplete}")

# Display excluded respondents for transparency
if num_incomplete > 0:
    print("\nRespondents excluded from FCM:")
    display(
    fcm_data.loc[
        incomplete_fcm_mask,
        [RESPONDENT_ID_COLUMN] + fcm_features
    ].reset_index(drop=True)
)

# Keep only respondents with complete core FCM values
fcm_data = fcm_data.loc[~incomplete_fcm_mask].copy()
fcm_interpretation_data = (
    fcm_interpretation_data
    .loc[~incomplete_fcm_mask]
    .copy()
)

print(f"\nRespondents retained for FCM clustering: {len(fcm_data)}")


# Only actual monetary variables receive Winsorization and log transformation.
monetary_cols = [
    '1. Average Monthly Gross Income',
    '2. Average Monthly Expenses',
    '3. Average Monthly Net Savings'
]

for col in monetary_cols:

    # Winsorization using the 1st and 99th percentiles
    lower_bound = fcm_data[col].quantile(0.01)
    upper_bound = fcm_data[col].quantile(0.99)

    fcm_data[col] = fcm_data[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    # Log transformation to reduce monetary skewness
    fcm_data[col] = np.log1p(fcm_data[col])

scaler = StandardScaler()

fcm_scaled_data = scaler.fit_transform(
    fcm_data[fcm_features]
)

df_fcm_scaled = pd.DataFrame(
    fcm_scaled_data,
    columns=fcm_features,
    index=fcm_data.index
)

print(
    "\nFCM data successfully prepared: "
    "incomplete core records excluded, "
    "monetary variables Winsorized and log-transformed, "
    "and all features standardized."
)

display(df_fcm_scaled.head())


This code cell prepares the data for the Fuzzy C-Means (FCM) clustering. It selects core FCM features, excludes respondents with missing values in these features, and then applies Winsorization and logarithmic transformation to monetary variables to reduce skewness. Finally, all features are standardized using `StandardScaler` to ensure they contribute equally to the clustering process.

### 3.1.2. Apply Fuzzy C-Means Clustering

Fuzzy C-Means clustering is applied using three clusters. The algorithm initially produces three unlabeled clusters and their fuzzy membership degrees. These clusters are subsequently interpreted and assigned Low, Medium, and High Financial Capacity labels in Section 3.1.3 based on their relative financial characteristics. The `skfuzzy.cmeans` function will be used to perform the clustering, yielding cluster centers, membership degrees, and the Fuzzy Partition Coefficient (FPC).


In [ ]:
# Transpose the data for skfuzzy.cmeans, which expects (features, samples)
C = 3

# Fuzziness coefficient used for the prototype.
# This value is fixed for all experiments to preserve reproducibility.
m = 1.5

# Perform Fuzzy C-Means clustering
cntr, u, u0, d, jm, p, fpc = fuzz.cmeans(
    df_fcm_scaled.T,
    C,
    m,
    error=0.005,
    maxiter=1000,
    seed=RANDOM_SEED
)

print(f"Fuzzy C-Means clustering completed with {C} clusters.")
print(f"Fuzzy Partition Coefficient (FPC): {fpc:.4f}")

# Dominant cluster for each respondent
cluster_membership = np.argmax(u, axis=0)

# Use fcm_data here because only these respondents were included in FCM.
df_fcm_results = pd.DataFrame({
    RESPONDENT_ID_COLUMN:
        fcm_data[RESPONDENT_ID_COLUMN].values,

    'Cluster_ID':
        cluster_membership
})

# Store fuzzy memberships
for i in range(C):
    df_fcm_results[f'Membership_Cluster_{i}'] = u[i]

display(df_fcm_results.head())


This code cell applies the Fuzzy C-Means clustering algorithm to the prepared and scaled FCM data. It initializes the clustering with 3 clusters and a fuzziness coefficient `m=1.5`. The `skfuzzy.cmeans` function performs the clustering, outputting cluster centers, membership degrees (`u`), and the Fuzzy Partition Coefficient (FPC), and stores the dominant cluster assignment and fuzzy memberships in `df_fcm_results`.

### 3.1.3. Interpret Clusters and Assign Labels

The three FCM clusters are initially unlabeled. To assign meaningful financial-capacity labels, the average original financial characteristics of each cluster are examined. Higher gross income and net savings indicate greater financial capacity, while higher monthly expenses and number of dependents represent greater financial burden.

These cluster-level characteristics are standardized and combined into a Capacity Interpretation Score. The cluster with the lowest score is labeled Low Financial Capacity, the intermediate cluster is labeled Medium Financial Capacity, and the highest-scoring cluster is labeled High Financial Capacity. This interpretation step does not alter the FCM clustering or membership degrees; it only assigns semantic labels to the clusters after clustering.


In [ ]:
# Add cluster assignment back to the raw/untransformed data for interpretation
fcm_interpretation_data['Cluster_ID'] = cluster_membership

# Calculate mean of original (RAW) features for each cluster
cluster_characteristics = fcm_interpretation_data.groupby('Cluster_ID')[fcm_features].mean()

print("--- Cluster Characteristics (Raw Financial Means) ---")
display(cluster_characteristics)

# Define the columns used for capacity interpretation
income_col = '1. Average Monthly Gross Income'
expense_col = '2. Average Monthly Expenses'
savings_col = '3. Average Monthly Net Savings'
dependents_col = 'Number of Dependents'

# These variables are used to determine the relative financial capacity represented by each FCM cluster.
capacity_variables = [
    income_col,
    savings_col,
    expense_col,
    dependents_col
]

# Standardization prevents variables with larger numeric scales, such as income, from dominating the cluster interpretation.
cluster_stds = (
    cluster_characteristics[capacity_variables]
    .std(ddof=0)
    .replace(0, 1e-9)
)

cluster_profile_z = (
    cluster_characteristics[capacity_variables]
    - cluster_characteristics[capacity_variables].mean()
) / cluster_stds

# Higher income and savings indicate greater capacity.
# Higher expenses and number of dependents indicate greater burden.
# This score is ONLY used to assign meaningful labels
# (Low / Medium / High) after FCM has already formed the clusters. It does not change the FCM clustering itself.
cluster_profile_z['Capacity_Interpretation_Score'] = (
    cluster_profile_z[income_col]
    + cluster_profile_z[savings_col]
    - cluster_profile_z[expense_col]
    - cluster_profile_z[dependents_col]
)

print("\n--- Standardized Cluster Profiles & Interpretation Score ---")
display(cluster_profile_z)

# Sort clusters by the new comprehensive capacity score
sorted_clusters_by_capacity = cluster_profile_z.sort_values('Capacity_Interpretation_Score').index.tolist()

# Assign labels based on the sorted capacity score
cluster_labels = {}
if len(sorted_clusters_by_capacity) == 3:
    cluster_labels[sorted_clusters_by_capacity[0]] = 'Low Financial Capacity'
    cluster_labels[sorted_clusters_by_capacity[1]] = 'Medium Financial Capacity'
    cluster_labels[sorted_clusters_by_capacity[2]] = 'High Financial Capacity'
else:
    print("WARNING: Unexpected number of clusters.")
    for i, cluster_id in enumerate(sorted_clusters_by_capacity):
        cluster_labels[cluster_id] = f'Cluster_{cluster_id}'

print("\n--- Assigned Cluster Labels ---")
for cluster_id, label in cluster_labels.items():
    print(f"Cluster {cluster_id}: {label}")

# Map numerical cluster IDs to meaningful labels in the results DataFrame
df_fcm_results['Financial_Capacity'] = df_fcm_results['Cluster_ID'].map(cluster_labels)

# Also add 'Financial_Capacity' to fcm_interpretation_data for reporting
fcm_interpretation_data['Financial_Capacity'] = fcm_interpretation_data['Cluster_ID'].map(cluster_labels)

# Rename membership columns to be consistent with labels
if len(sorted_clusters_by_capacity) == 3:
    df_fcm_results = df_fcm_results.rename(columns={
        f'Membership_Cluster_{sorted_clusters_by_capacity[0]}': 'Membership_Low',
        f'Membership_Cluster_{sorted_clusters_by_capacity[1]}': 'Membership_Medium',
        f'Membership_Cluster_{sorted_clusters_by_capacity[2]}': 'Membership_High'
    })

# Final output DataFrame for FCM
df_fcm_final_output = df_fcm_results[[RESPONDENT_ID_COLUMN, 'Financial_Capacity', 'Membership_Low', 'Membership_Medium', 'Membership_High']].copy()

print("\nFinal FCM Output (First five rows):")
display(df_fcm_final_output.head())


This code cell interprets the unlabeled FCM clusters by examining the mean of the original (raw) financial characteristics for each cluster. It standardizes these characteristics to create a 'Capacity_Interpretation_Score'. Clusters are then sorted by this score and assigned meaningful labels: 'Low Financial Capacity', 'Medium Financial Capacity', and 'High Financial Capacity'. These labels are then mapped back to the `df_fcm_results` DataFrame.

### 3.1.4. Financial Capacity Module Report

This report summarizes the results from the Fuzzy C-Means clustering for financial capacity analysis.


In [ ]:
print("--- Financial Capacity Module Report ---")
print(f"Number of respondents clustered: {len(df_fcm_final_output)}")
print(f"Number of clusters: {C}")
print(f"Fuzzy Partition Coefficient (FPC): {fpc:.4f}")

print("\nDistribution of Financial Capacity Labels:")
display(df_fcm_final_output['Financial_Capacity'].value_counts())

print("\nCluster Characteristics (Mean of RAW Original Features for each labeled cluster):")
# We use fcm_interpretation_data here so the report prints actual Peso values
report_characteristics = fcm_interpretation_data.groupby('Financial_Capacity')[fcm_features].mean()
display(report_characteristics)

print("\nFCM module successfully executed. The 'df_fcm_final_output' DataFrame contains the respondent's financial capacity and membership degrees.")


This code cell generates a summary report for the Financial Capacity Module. It prints the number of clustered respondents, the number of clusters, and the Fuzzy Partition Coefficient (FPC). It also displays the distribution of financial capacity labels and the mean of raw financial features for each labeled cluster, confirming the successful execution of the FCM module.

### 3.1.5. Financial Capacity Results
This subsection summarizes the final Fuzzy C-Means clustering results for reporting. It presents the number and percentage of respondents assigned to each Financial Capacity category, the raw financial characteristics of the resulting clusters, the Capacity Interpretation Score used for semantic labeling, and a representative respondent with a strong fuzzy membership for each category.

The additional expense-to-income and savings-to-income ratios shown below are descriptive measures used only to assist interpretation of the resulting clusters. They are not clustering inputs and do not alter the FCM algorithm.


In [ ]:
CAPACITY_ORDER = [
    'Low Financial Capacity',
    'Medium Financial Capacity',
    'High Financial Capacity'
]


# COUNTS AND PERCENTAGES

capacity_counts = (
    df_fcm_final_output[
        'Financial_Capacity'
    ]
    .value_counts()
    .reindex(
        CAPACITY_ORDER,
        fill_value=0
    )
)


capacity_percentages = (
    df_fcm_final_output[
        'Financial_Capacity'
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .reindex(
        CAPACITY_ORDER,
        fill_value=0
    )
)


paper_fcm_distribution = pd.DataFrame({
    'Respondents':
        capacity_counts,

    'Percentage':
        capacity_percentages.round(2)
})


print(
    "\n=== FINANCIAL CAPACITY DISTRIBUTION ==="
)

display(
    paper_fcm_distribution
)


# RAW CLUSTER CHARACTERISTICS

paper_fcm_characteristics = (
    fcm_interpretation_data
    .groupby(
        'Financial_Capacity'
    )[
        fcm_features
    ]
    .mean()
    .reindex(
        CAPACITY_ORDER
    )
)


# Descriptive ratios only — NOT FCM inputs.
paper_fcm_characteristics[
    'Expense_to_Income_%'
] = (
    paper_fcm_characteristics[
        '2. Average Monthly Expenses'
    ]
    /
    paper_fcm_characteristics[
        '1. Average Monthly Gross Income'
    ]
    * 100
)


paper_fcm_characteristics[
    'Savings_to_Income_%'
] = (
    paper_fcm_characteristics[
        '3. Average Monthly Net Savings'
    ]
    /
    paper_fcm_characteristics[
        '1. Average Monthly Gross Income'
    ]
    * 100
)


# CAPACITY INTERPRETATION SCORE

capacity_score_mapping = {}

for cluster_id, capacity_label in (
    cluster_labels.items()
):

    capacity_score_mapping[
        capacity_label
    ] = (
        cluster_profile_z.loc[
            cluster_id,
            'Capacity_Interpretation_Score'
        ]
    )


paper_fcm_characteristics[
    'Capacity_Interpretation_Score'
] = pd.Series(
    capacity_score_mapping
)


paper_fcm_characteristics = (
    paper_fcm_characteristics
    .round(2)
)


print(
    "\n=== FINANCIAL CAPACITY CLUSTER CHARACTERISTICS ==="
)

display(
    paper_fcm_characteristics
)


# REPRESENTATIVE RESPONDENT PER CAPACITY

membership_column_map = {

    'Low Financial Capacity':
        'Membership_Low',

    'Medium Financial Capacity':
        'Membership_Medium',

    'High Financial Capacity':
        'Membership_High'
}


representative_fcm_records = []


for capacity_label in CAPACITY_ORDER:

    membership_col = (
        membership_column_map[
            capacity_label
        ]
    )

    candidates = (
        df_fcm_final_output[
            df_fcm_final_output[
                'Financial_Capacity'
            ]
            == capacity_label
        ]
    )

    if candidates.empty:
        continue

    representative_index = (
        candidates[
            membership_col
        ]
        .idxmax()
    )

    representative = (
        candidates.loc[
            representative_index
        ]
    )

    respondent_id = (
        representative[
            RESPONDENT_ID_COLUMN
        ]
    )

    raw_profile = (
        fcm_interpretation_data.loc[
            fcm_interpretation_data[
                RESPONDENT_ID_COLUMN
            ]
            == respondent_id
        ]
        .iloc[0]
    )

    representative_fcm_records.append({

        RESPONDENT_ID_COLUMN:
            respondent_id,

        'Financial_Capacity':
            capacity_label,

        'Dominant_Membership':
            representative[
                membership_col
            ],

        'Number_of_Dependents':
            raw_profile[
                'Number of Dependents'
            ],

        'Gross_Income':
            raw_profile[
                '1. Average Monthly Gross Income'
            ],

        'Monthly_Expenses':
            raw_profile[
                '2. Average Monthly Expenses'
            ],

        'Net_Savings':
            raw_profile[
                '3. Average Monthly Net Savings'
            ]
    })


paper_fcm_representatives = (
    pd.DataFrame(
        representative_fcm_records
    )
    .round(3)
)


print(
    "\n=== REPRESENTATIVE FCM RESPONDENTS ==="
)

display(
    paper_fcm_representatives
)


# DISTRIBUTION FIGURE

plt.figure(
    figsize=(8, 5)
)

paper_fcm_distribution[
    'Respondents'
].plot(
    kind='bar'
)

plt.title(
    'Distribution of Financial Capacity Profiles'
)

plt.xlabel(
    'Financial Capacity'
)

plt.ylabel(
    'Number of Respondents'
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        'figure_fcm_capacity_distribution.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

plt.show()


# SAVE TABLES

paper_fcm_distribution.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_fcm_distribution.csv'
    )
)

paper_fcm_characteristics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_fcm_characteristics.csv'
    )
)

paper_fcm_representatives.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_fcm_representatives.csv'
    ),
    index=False
)


This code cell summarizes the final Fuzzy C-Means clustering results for reporting. It calculates the counts and percentages of respondents in each financial capacity category, displays raw cluster characteristics including expense-to-income and savings-to-income ratios, shows the Capacity Interpretation Score, and identifies representative respondents for each capacity level. It also generates and saves a bar plot of the financial capacity distribution.

## 3.2 MARKET FORECASTING MODULE: PROPHET

This module focuses on forecasting the Philippine Stock Exchange Index (PSEI) to generate market signals. We will use the Prophet library, which is well-suited for time series forecasting, especially with data exhibiting trends and seasonality.

The steps involve:
1.  Retrieving historical PSEI data.
2.  Preparing the time series data in Prophet's required format.
3.  Training a Prophet model.
4.  Generating a future forecast.
5.  Converting the forecast into defined market signal categories (Positive, Neutral, Negative, Volatile).

### 3.2.1. Retrieve Historical PSEI Market Data

Historical PSEi closing-price data are provided through a manually downloaded CSV obtained from a reputable financial data source (investing.com) Manual upload is used to ensure reproducibility of the exact dataset used during experimentation.


In [ ]:
import pandas as pd
from google.colab import files
import io

print("Please upload your manually downloaded PSEi historical data CSV file:")
print("(Make sure it has 'Date' and 'Close' or 'Price' columns)")

# Upload the CSV file
uploaded_market = files.upload()

# Get the filename
market_file_name = list(uploaded_market.keys())[0]

# Read the CSV into a DataFrame
psei_data = pd.read_csv(io.BytesIO(uploaded_market[market_file_name]))

print(f"\nHistorical PSEI data retrieved from '{market_file_name}'.")
print(f"Number of historical data points: {len(psei_data)}")
display(psei_data.head())


This code cell prompts the user to upload a CSV file containing historical PSEi market data. It then reads the uploaded file into a pandas DataFrame, `psei_data`, and displays the first few rows and the total number of data points, confirming successful data retrieval for market forecasting.

### 3.2.2. Prepare Time Series Data for Prophet

Prophet requires the time series data to be in a specific format: a DataFrame with two columns, `ds` (datetime) and `y` (numerical value to be forecasted). We will prepare the `Close` price of the PSEI data in this format.


In [ ]:
# Check if data is empty
if psei_data.empty:
    raise ValueError("No historical data found in the CSV.")

# Yahoo Finance uses "Close". Investing.com uses "Price". Renamed Price to Close
if 'Price' in psei_data.columns and 'Close' not in psei_data.columns:
    psei_data.rename(columns={'Price': 'Close'}, inplace=True)

# Select relevant columns
df_prophet = psei_data[['Date', 'Close']].copy()

# Rename columns to 'ds' and 'y' as required by Prophet
df_prophet.rename(columns={'Date': 'ds', 'Close': 'y'}, inplace=True)

# Ensure 'ds' is datetime
df_prophet['ds'] = pd.to_datetime(df_prophet['ds'])

# Ensure 'y' is numeric.
if df_prophet['y'].dtype == 'object':
    df_prophet['y'] = df_prophet['y'].str.replace(',', '').astype(float)
else:
    df_prophet['y'] = pd.to_numeric(df_prophet['y'])

df_prophet = df_prophet.sort_values(by='ds').reset_index(drop=True)

# Check for missing values in 'y' and fill with previous day's price (Forward Fill)
if df_prophet['y'].isnull().any():
    print("Warning: Missing values found in 'y' column. Filling with forward fill.")
    df_prophet['y'] = df_prophet['y'].ffill()

print("Data prepared and sorted for Prophet.")
display(df_prophet.head())


This code cell prepares the historical PSEi data for the Prophet model. It renames the 'Date' and 'Close' (or 'Price') columns to 'ds' and 'y' respectively, converts 'ds' to datetime objects and 'y' to numeric, sorts the data by date, and handles any missing values in 'y' by forward filling. This ensures the data is in the required format for Prophet.

### 3.2.3. Plot Historical Market Data

Visualizing the historical `Close` prices will help us understand the trends, seasonality, and any anomalies present in the PSEI data.


In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df_prophet['ds'], df_prophet['y'])
plt.title('Historical PSEi Closing Index Level')
plt.xlabel('Date')
plt.ylabel('PSEi Index Level')
plt.grid(True)
plt.tight_layout()
plt.show()

# Optional: plot using Prophet's built-in plot function for initial visual check
# m = Prophet()
# m.fit(df_prophet) # Temporarily fit to use plot function
# fig = m.plot(df_prophet) # This will only plot historical, not future


This code cell visualizes the historical PSEi closing index levels. It plots the `ds` (date) against `y` (index level) from the `df_prophet` DataFrame, providing a clear line plot of the market's historical trends. This helps in understanding the data before forecasting.

### 3.2.4. Train Prophet Model and Generate Forecast

We will now initialize and train the Prophet model using the prepared historical data. Then, we will generate a forecast for a specified future horizon. The Prophet model is trained using the historical PSEi closing-price series and generates a 30-business-day forecast. Business-day frequency is used to exclude weekends from the forecast horizon.


In [ ]:
# Initialize Prophet model
# You can add parameters like daily_seasonality=True, weekly_seasonality=True, etc.
# based on the nature of your time series and analysis.
m = Prophet(seasonality_mode='multiplicative',
            changepoint_prior_scale=0.05,
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=True)

# Fit the model with historical data
m.fit(df_prophet)

# Define the forecasting horizon
future_horizon_days = FORECAST_HORIZON_DAYS # Forecast for the next 30 days
future = m.make_future_dataframe(periods=future_horizon_days,freq='B')

# Generate forecast
forecast = m.predict(future)

print(f"Prophet model trained and forecast generated for the next {future_horizon_days} days.")
print("Forecast DataFrame columns: ds, trend, yhat_lower, yhat_upper, trend_lower, trend_upper, yhat.")
display(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

def get_real_market_forecast():
    """
    Calculates the projected percentage change in the PSEi
    over the configured forecast horizon.
    """
    # Get the very last known real index level
    current_index_level = df_prophet.iloc[-1]['y']

    # Get the very last predicted future index level
    predicted_index_level = forecast.iloc[-1]['yhat']

    # Calculate growth rate
    growth_rate = (predicted_index_level - current_index_level) / current_index_level

    print(f"\n--- MARKET FORECAST SUMMARY ---")
    print(f"Current PSEi Index Level: {current_index_level:,.2f}")
    print(f"Predicted PSEi Index Level ({future_horizon_days} business days): {predicted_index_level:,.2f}")
    print(f"Projected Change: {growth_rate * 100:.2f}%\n")

    return growth_rate

market_signal = get_real_market_forecast()


This code cell initializes and trains the Prophet model with the historical PSEi data. It configures the model with `seasonality_mode='multiplicative'` and `yearly_seasonality=True`, then fits it to `df_prophet`. It then generates a 30-business-day forecast and calculates the projected growth rate of the PSEi, summarizing the current and predicted index levels.

### 3.2.5. Plot Market Forecast

Visualizing the forecast along with the historical data helps in understanding the predicted future trend and the uncertainty range.


In [ ]:
fig1 = m.plot(forecast)
plt.title(f'PSEi Forecast for Next {future_horizon_days} Days')
plt.xlabel('Date')
plt.ylabel('PSEi Index Level')
plt.show()

fig2 = m.plot_components(forecast)
plt.show()


This code cell visualizes the Prophet forecast. It generates a plot showing the historical PSEi data alongside the predicted future trend and its uncertainty intervals. It also plots the individual components of the forecast (trend, yearly seasonality), providing a comprehensive view of the model's predictions.

### 3.2.6. Calculate Market Signal Categories

The Prophet forecast is converted into Positive, Neutral, Negative, or Volatile market signals. A projected increase of at least 1% is classified as Positive, while a projected decrease of at least 1% is classified as Negative. A forecast-interval width greater than 6% of the predicted value is classified as Volatile and takes precedence over directional classification. Forecasts not meeting these conditions are classified as Neutral. In this prototype, the Volatile label represents high forecast uncertainty rather than directly measured historical market volatility.


In [ ]:
def categorize_market_signal(growth_rate, forecast_df):
    """
    Evaluates the growth rate and model uncertainty to categorize the market.
    """

    # calculate Volatility (uncertainty spread)
    # we look at the last day of the forecast to see how wide the confidence interval got
    last_forecast = forecast_df.iloc[-1]
    price_spread = last_forecast['yhat_upper'] - last_forecast['yhat_lower']
    uncertainty_ratio = price_spread / last_forecast['yhat']

    print(f"--- SIGNAL CLASSIFICATION METRICS ---")
    print(f"Projected Growth Rate: {growth_rate * 100:.2f}%")
    print(f"Forecast Uncertainty Ratio: {uncertainty_ratio * 100:.2f}%")

    # apply Classification Logic
    # we check volatility FIRST. If the market is too unpredictable, it's Volatile.
    if uncertainty_ratio > FORECAST_UNCERTAINTY_THRESHOLD:
        signal = "Volatile"
        explanation = "The Prophet forecast interval is relatively wide, indicating high forecast uncertainty."

    elif growth_rate >= POSITIVE_THRESHOLD:
        signal = "Positive"
        explanation = "The market shows a stable upward trend."

    elif growth_rate <= NEGATIVE_THRESHOLD:
        signal = "Negative"
        explanation = "The market is showing a downward trend."

    else:
        signal = "Neutral"
        explanation = "The market is relatively flat with no strong upward or downward momentum."

    print(f"\nFINAL MARKET SIGNAL: [ {signal.upper()} ]")
    print(f"Reasoning: {explanation}")

    return signal

# run growth rate
final_market_signal = categorize_market_signal(market_signal, forecast)


This code cell defines and applies a function, `categorize_market_signal`, to classify the market based on the Prophet forecast. It calculates the projected growth rate and an uncertainty ratio. Based on predefined thresholds, it assigns a market signal of 'Positive', 'Neutral', 'Negative', or 'Volatile', with 'Volatile' taking precedence if forecast uncertainty is high.

### 3.2.7. Market Forecast Results

This subsection presents the primary Prophet output used in the study, including the current PSEi index level, forecasted index level after the 30-business-day horizon, projected percentage change, forecast uncertainty ratio, and resulting categorical Market Signal.

The forecast plot focuses on the most recent historical observations and the future forecast horizon to improve readability for presentation in the Results and Discussion chapter.


In [ ]:
# MARKET FORECAST RESULTS

last_forecast_row = (
    forecast.iloc[-1]
)


current_index_level = (
    df_prophet.iloc[-1]['y']
)


predicted_index_level = (
    last_forecast_row['yhat']
)


projected_growth_rate = (
    predicted_index_level
    - current_index_level
) / current_index_level


forecast_interval_width = (
    last_forecast_row['yhat_upper']
    - last_forecast_row['yhat_lower']
)


forecast_uncertainty_ratio = (
    forecast_interval_width
    / last_forecast_row['yhat']
)


paper_market_summary = pd.DataFrame({

    'Metric': [
        'Last Historical Date',
        'Current PSEi Index Level',
        'Forecast End Date',
        'Predicted PSEi Index Level',
        'Projected Change (%)',
        'Forecast Uncertainty (%)',
        'Final Market Signal'
    ],

    'Value': [
        df_prophet.iloc[-1]['ds'],
        round(
            current_index_level,
            2
        ),
        last_forecast_row['ds'],
        round(
            predicted_index_level,
            2
        ),
        round(
            projected_growth_rate
            * 100,
            2
        ),
        round(
            forecast_uncertainty_ratio
            * 100,
            2
        ),
        final_market_signal
    ]
})


print(
    "\n=== PAPER-READY PROPHET FORECAST SUMMARY ==="
)

display(
    paper_market_summary
)


# FUTURE FORECAST TABLE

last_historical_date = (
    df_prophet['ds'].max()
)


paper_future_forecast = (
    forecast[
        forecast['ds']
        > last_historical_date
    ][
        [
            'ds',
            'yhat',
            'yhat_lower',
            'yhat_upper'
        ]
    ]
    .copy()
)


paper_future_forecast.columns = [
    'Date',
    'Predicted_Index',
    'Lower_Bound',
    'Upper_Bound'
]


paper_future_forecast[
    [
        'Predicted_Index',
        'Lower_Bound',
        'Upper_Bound'
    ]
] = (
    paper_future_forecast[
        [
            'Predicted_Index',
            'Lower_Bound',
            'Upper_Bound'
        ]
    ]
    .round(2)
)


print(
    "\n=== 30-BUSINESS-DAY FORECAST ==="
)

display(
    paper_future_forecast
)


# PAPER-READY FORECAST FIGURE

recent_history = (
    df_prophet.tail(252)
)


plt.figure(
    figsize=(12, 6)
)


plt.plot(
    recent_history['ds'],
    recent_history['y'],
    label='Historical PSEi'
)


plt.plot(
    paper_future_forecast['Date'],
    paper_future_forecast[
        'Predicted_Index'
    ],
    label='Prophet Forecast'
)


plt.fill_between(
    paper_future_forecast['Date'],
    paper_future_forecast[
        'Lower_Bound'
    ],
    paper_future_forecast[
        'Upper_Bound'
    ],
    alpha=0.2,
    label='Forecast Interval'
)


plt.axvline(
    last_historical_date,
    linestyle='--',
    label='Forecast Start'
)


plt.title(
    'PSEi Historical Index and '
    '30-Business-Day Prophet Forecast'
)

plt.xlabel(
    'Date'
)

plt.ylabel(
    'PSEi Index Level'
)

plt.legend()

plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        'figure_prophet_forecast.png'
    ),
    dpi=300,
    bbox_inches='tight'
)


plt.show()


paper_market_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_market_summary.csv'
    ),
    index=False
)


paper_future_forecast.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_future_forecast.csv'
    ),
    index=False
)


This code cell presents a paper-ready summary of the Prophet market forecast. It compiles key metrics such as the current and predicted PSEi levels, projected percentage change, forecast uncertainty, and the final market signal into a DataFrame. It also generates a focused plot of recent history and the future forecast, and saves these results to CSV files and a PNG image.

## 3.3 BEHAVIORAL ANALYSIS MODULE: XLM-ROBERTA

### 3.3.1. Open-Ended Response Inspection

This section inspects the completeness of the open-ended survey responses prior to behavioral classification. The final Behavioral Analysis Module directly evaluates Question 7 and Question 9 because these questions specifically measure investment risk preference and behavioral reaction to financial loss. Each question is classified independently using XLM-RoBERTa zero-shot classification before the two categorical results are combined using explicit decision rules.


In [ ]:
print("--- Open-Ended Response Completeness Inspection ---")
missing_xlmr = df_xlmr[xlmr_cols].isnull().sum()
missing_xlmr = missing_xlmr[missing_xlmr > 0]

if not missing_xlmr.empty:
    print("Columns with missing open-ended responses:")
    display(missing_xlmr)
else:
    print("No missing values found in open-ended response columns.")

# Display a sample of cleaned text to confirm readiness for XLM-RoBERTa
print("\nSample of cleaned open-ended responses:")
for col in xlmr_cols:
    print(f"- {col}:")
    display(df_xlmr[col].head(2).to_frame())


This code cell inspects the completeness of open-ended survey responses (Q7 and Q9) for the Behavioral Analysis Module. It checks for missing values in `xlmr_cols` within the `df_xlmr` DataFrame and displays any columns with missing data. It also presents a sample of the cleaned text to confirm readiness for the XLM-RoBERTa model.

### 3.3.2. XLM-RoBERTa Zero-Shot Behavioral Classification

The Behavioral Analysis Module evaluates the two open-ended survey questions most directly related to investment risk tolerance.

Question 7 measures the respondent's stated preference between traditional banking or safer financial options, diversified or moderate-risk investment, and high-risk/high-reward investment.

Question 9 measures the respondent's behavioral reaction to a hypothetical investment loss, distinguishing between risk-avoidant behavior, waiting or observing behavior, and willingness to continue accepting substantial risk.

The two responses are classified independently using the pretrained XLM-RoBERTa zero-shot classification model. Their final categorical results are subsequently combined through explicit decision rules. Raw XLM-RoBERTa probability scores are retained for diagnostic interpretation but are not averaged to determine the final Behavioral Profile.

The final Behavioral Profile is categorized as Low Risk Tolerance, Neutral, or High Risk Tolerance.


In [ ]:
# 3.3.2 XLM-RoBERTa BEHAVIORAL RISK CLASSIFICATION


print("Loading XLM-RoBERTa Zero-Shot Classifier...")

classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)


# QUESTIONS DIRECTLY RELATED TO INVESTMENT RISK

# Q7 measures the respondent's stated preference between
# safer/traditional financial options and higher-risk investments.
risk_preference_col = (
    '7. How do you feel about investing in high-risk, high-reward '
    'platforms (like Crypto or Stocks) compared to keeping your '
    'money in a traditional bank? (Do you strictly prefer one '
    'over the other, or do you prefer to put half in the bank '
    'and half in investments?)'
)

# Q9 measures the respondent's behavioral reaction to
# a substantial hypothetical investment loss.
loss_reaction_col = (
    "9. Imagine you invested ₱10,000, and after one week, its value "
    "dropped to ₱7,000. What would you do and how would you feel? "
    "(e.g., Would you panic and sell everything, buy more because "
    "it's cheap, or just observe and wait?)"
)


# QUESTION 7 LABELS

# LOW:
# Prefers traditional banking / safer financial options.

# NEUTRAL:
# Prefers diversification, mutual funds, or a mixture of
# safer and riskier financial options.

# HIGH:
# Explicitly prefers speculative/high-risk investments such as
# individual stocks or cryptocurrency for potentially higher returns.

q7_labels = {

    (
        "prefers traditional banking, savings accounts "
        "or other low-risk financial options"
    ):
        "Low Risk Tolerance",

    (
        "prefers diversified or moderate-risk investments "
        "such as mutual funds, or a mixture of bank savings "
        "and investments"
    ):
        "Neutral",

    (
        "prefers speculative or high-risk investments "
        "such as individual stocks or cryptocurrency "
        "for potentially higher returns"
    ):
        "High Risk Tolerance"
}


# QUESTION 9 LABELS

# LOW:
# Panic, sell, or avoid further investing after a loss.

# NEUTRAL:
# Hold, observe, or wait before deciding.

# HIGH:
# Buy more or continue accepting substantial investment risk.

q9_labels = {

    (
        "selling the investment or avoiding further investment"
    ):
        "Low Risk Tolerance",

    (
        "holding the investment and waiting to see what happens"
    ):
        "Neutral",

    (
        "buying more or continuing to take substantial "
        "investment risk"
    ):
        "High Risk Tolerance"
}


# Final behavioral classes used by the rule-based system
RISK_PROFILES = [
    "Low Risk Tolerance",
    "Neutral",
    "High Risk Tolerance"
]


# INVALID / NON-INFORMATIVE RESPONSES

INVALID_TEXT_RESPONSES = {
    "",
    "-",
    "--",
    "---",
    "n/a",
    "na",
    "not applicable",
    "no answer"
}


# CLASSIFY ONE QUESTION

def classify_risk_question(
    text,
    label_map,
    hypothesis_template
):
    """
    Classifies one open-ended behavioral response using
    XLM-RoBERTa zero-shot classification.

    Returns:
        {
            'profile': predicted Low / Neutral / High profile,
            'confidence': winning zero-shot score,
            'scores': dictionary containing all three profile scores
        }

    Returns None if the response is missing or non-informative.
    """

    # Missing value
    if pd.isna(text):
        return None

    text = str(text).strip()

    # Empty after cleaning
    if not text:
        return None

    # Detect placeholder/non-informative responses
    if text.lower() in INVALID_TEXT_RESPONSES:
        return None


    # Candidate statements used by XLM-R
    candidate_labels = list(
        label_map.keys()
    )


    # Run zero-shot classification
    result = classifier(
        text,
        candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False
    )


    # Initialize scores for the three final behavioral profiles
    profile_scores = {
        profile: 0.0
        for profile in RISK_PROFILES
    }


    # Convert XLM-R candidate scores into
    # Low / Neutral / High profile scores.
    for label, score in zip(
        result['labels'],
        result['scores']
    ):

        mapped_profile = (
            label_map[label]
        )

        profile_scores[
            mapped_profile
        ] = float(score)


    # Determine the winning profile for this question
    predicted_profile = max(
        profile_scores,
        key=profile_scores.get
    )

    confidence = (
        profile_scores[
            predicted_profile
        ]
    )


    return {
        'profile':
            predicted_profile,

        'confidence':
            confidence,

        'scores':
            profile_scores
    }


# COMBINE Q7 AND Q9

def combine_risk_profiles(
    q7_result,
    q9_result
):
    """
    Combines the independently classified Q7 and Q9 behavioral
    profiles using transparent decision rules.

    IMPORTANT:
    Raw XLM-R probabilities are NOT averaged to determine the
    final behavioral class.

    Rules:

    1. If both questions agree:
       -> use the agreed profile.

    2. If Q7 is Neutral but Q9 is Low/High:
       -> Q9 resolves the classification.

    3. If Q9 is Neutral but Q7 is Low/High:
       -> Q7 resolves the classification.

    4. If one question is Low and the other is High:
       -> classify as Neutral because the behavioral evidence
          directly conflicts.

    5. If only one valid response is available:
       -> use that response.

    6. If neither is usable:
       -> no behavioral profile.
    """

    q7_profile = (
        q7_result['profile']
        if q7_result is not None
        else None
    )

    q9_profile = (
        q9_result['profile']
        if q9_result is not None
        else None
    )


    # No usable behavioral answers

    if (
        q7_profile is None
        and q9_profile is None
    ):

        return (
            np.nan,
            np.nan,
            "No usable Q7 or Q9 response"
        )


    # Only Q7 is available

    if q9_profile is None:

        return (
            q7_profile,
            q7_result['confidence'],
            "Q7 only"
        )


    # Only Q9 is available

    if q7_profile is None:

        return (
            q9_profile,
            q9_result['confidence'],
            "Q9 only"
        )


    # Q7 and Q9 agree

    if q7_profile == q9_profile:

        final_confidence = np.mean([
            q7_result['confidence'],
            q9_result['confidence']
        ])

        return (
            q7_profile,
            final_confidence,
            "Q7 and Q9 agree"
        )


    # Q7 is Neutral
    #
    # Use Q9 because Q9 provides the clearer directional
    # Low/High behavioral evidence.

    if q7_profile == "Neutral":

        return (
            q9_profile,
            q9_result['confidence'],
            "Q9 resolved Neutral Q7"
        )


    # Q9 is Neutral
    #
    # Preserve the respondent's explicit stated risk
    # preference from Q7.

    if q9_profile == "Neutral":

        return (
            q7_profile,
            q7_result['confidence'],
            "Q7 resolved Neutral Q9"
        )


    # Direct Low-vs-High conflict
    #
    # Example:
    # Q7 = Low
    # Q9 = High
    #
    # The behavioral evidence is contradictory, therefore
    # classify the overall respondent as Neutral.
    #
    # We intentionally do NOT assign a numerical confidence
    # because neither classifier predicted Neutral.

    return (
        "Neutral",
        np.nan,
        "Low/High behavioral conflict -> Neutral"
    )


# CLASSIFY ONE RESPONDENT

def classify_behavioral_profile(row):
    """
    Classifies Q7 and Q9 separately, then combines the two
    classifications using the transparent behavioral rules above.
    """

    q7_text = row[
        risk_preference_col
    ]

    q9_text = row[
        loss_reaction_col
    ]


    # QUESTION 7

    q7_result = classify_risk_question(
        q7_text,
        q7_labels,
        "The respondent {}."
    )


    # QUESTION 9

    q9_result = classify_risk_question(
        q9_text,
        q9_labels,
        (
            "After a large investment loss, "
            "the respondent would choose {}."
        )
    )


    # COMBINE THE TWO BEHAVIORAL RESULTS

    (
        final_profile,
        final_confidence,
        decision_basis
    ) = combine_risk_profiles(
        q7_result,
        q9_result
    )


    # Individual question classifications
    q7_profile = (
        q7_result['profile']
        if q7_result is not None
        else np.nan
    )

    q9_profile = (
        q9_result['profile']
        if q9_result is not None
        else np.nan
    )


    # DIAGNOSTIC SCORE AVERAGES

    # We still retain these values because they are useful for
    # researchers inspecting how XLM-R behaved.

    valid_results = [
        result
        for result in [
            q7_result,
            q9_result
        ]
        if result is not None
    ]


    if valid_results:

        diagnostic_scores = {}

        for profile in RISK_PROFILES:

            diagnostic_scores[profile] = np.mean(
                [
                    result['scores'][profile]
                    for result in valid_results
                ]
            )

    else:

        diagnostic_scores = {
            "Low Risk Tolerance": np.nan,
            "Neutral": np.nan,
            "High Risk Tolerance": np.nan
        }


    return pd.Series([
        final_profile,
        final_confidence,

        q7_profile,
        q9_profile,

        diagnostic_scores[
            "Low Risk Tolerance"
        ],

        diagnostic_scores[
            "Neutral"
        ],

        diagnostic_scores[
            "High Risk Tolerance"
        ],

        decision_basis
    ])


# RUN BEHAVIORAL CLASSIFICATION

df_xlmr_results = (
    df_xlmr[
        [RESPONDENT_ID_COLUMN]
    ]
    .copy()
)


df_xlmr_results[
    [
        'Behavioral_Profile',
        'Behavioral_Confidence',

        'Q7_Risk_Profile',
        'Q9_Loss_Profile',

        'Risk_Score_Low',
        'Risk_Score_Neutral',
        'Risk_Score_High',

        'Behavioral_Decision_Basis'
    ]
] = (
    df_xlmr.apply(
        classify_behavioral_profile,
        axis=1
    )
)


# REPORT

print(
    "\nBehavioral Classification Complete."
)


unclassified_xlmr = (
    df_xlmr_results[
        'Behavioral_Profile'
    ]
    .isna()
)


print(
    "\nRespondents without a valid "
    "behavioral profile:",
    unclassified_xlmr.sum()
)


if unclassified_xlmr.any():

    print(
        "\nUnclassified Respondents:"
    )

    display(
        df_xlmr_results.loc[
            unclassified_xlmr
        ]
    )


print(
    "\nBehavioral Profile Distribution:"
)

print(
    df_xlmr_results[
        'Behavioral_Profile'
    ]
    .value_counts()
)


print(
    "\nBehavioral Profile Percentages:"
)

print(
    df_xlmr_results[
        'Behavioral_Profile'
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)


print(
    "\nBehavioral Decision Basis Distribution:"
)

print(
    df_xlmr_results[
        'Behavioral_Decision_Basis'
    ]
    .value_counts(
        dropna=False
    )
)


# INSPECT INDIVIDUAL RESPONDENTS

def inspect_behavioral_classification(
    respondent_id
):

    source_row = df_xlmr.loc[
        df_xlmr[
            RESPONDENT_ID_COLUMN
        ] == respondent_id
    ]

    result_row = (
        df_xlmr_results.loc[
            df_xlmr_results[
                RESPONDENT_ID_COLUMN
            ] == respondent_id
        ]
    )


    if source_row.empty:

        print(
            f"Respondent "
            f"{respondent_id} not found."
        )

        return


    print(
        f"\n=== RESPONDENT "
        f"{respondent_id} ==="
    )


    print(
        "\nQ7 Risk Preference:"
    )

    print(
        source_row.iloc[0][
            risk_preference_col
        ]
    )


    print(
        "\nQ9 Loss Reaction:"
    )

    print(
        source_row.iloc[0][
            loss_reaction_col
        ]
    )


    print(
        "\nClassification:"
    )

    display(
        result_row[
            [
                RESPONDENT_ID_COLUMN,

                'Q7_Risk_Profile',
                'Q9_Loss_Profile',

                'Behavioral_Profile',
                'Behavioral_Confidence',

                'Risk_Score_Low',
                'Risk_Score_Neutral',
                'Risk_Score_High',

                'Behavioral_Decision_Basis'
            ]
        ]
    )


# CHECKS

inspect_behavioral_classification(1)
inspect_behavioral_classification(2)
inspect_behavioral_classification(5)


This comprehensive code cell performs XLM-RoBERTa zero-shot classification on the open-ended survey responses (Q7 and Q9) to determine each respondent's behavioral risk profile. It defines labels for different risk preferences and reactions to loss, classifies each question independently, and then combines these results using a set of explicit decision rules to assign a final 'Behavioral_Profile' (Low, Neutral, High Risk Tolerance). It also provides diagnostic scores and reports the distribution of these profiles.

### 3.3.3 BEHAVIORAL RESULTS


In [ ]:
# BEHAVIORAL RESULTS

BEHAVIOR_ORDER = [
    'Low Risk Tolerance',
    'Neutral',
    'High Risk Tolerance'
]


# FINAL PROFILE DISTRIBUTION

behavior_counts = (
    df_xlmr_results[
        'Behavioral_Profile'
    ]
    .value_counts()
    .reindex(
        BEHAVIOR_ORDER,
        fill_value=0
    )
)


behavior_percentages = (
    df_xlmr_results[
        'Behavioral_Profile'
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .reindex(
        BEHAVIOR_ORDER,
        fill_value=0
    )
)


paper_behavior_distribution = (
    pd.DataFrame({

        'Respondents':
            behavior_counts,

        'Percentage':
            behavior_percentages.round(2)
    })
)


print(
    "\n=== BEHAVIORAL PROFILE DISTRIBUTION ==="
)

display(
    paper_behavior_distribution
)


# Q7 × Q9 CLASSIFICATION TABLE

paper_behavior_crosstab = (
    pd.crosstab(
        df_xlmr_results[
            'Q7_Risk_Profile'
        ],
        df_xlmr_results[
            'Q9_Loss_Profile'
        ],
        margins=True
    )
)


print(
    "\n=== Q7 × Q9 CLASSIFICATION CROSS-TABULATION ==="
)

display(
    paper_behavior_crosstab
)


# REPRESENTATIVE RESPONDENT PER PROFILE

behavior_source = (
    df_xlmr[
        [
            RESPONDENT_ID_COLUMN,
            risk_preference_col,
            loss_reaction_col
        ]
    ]
    .merge(
        df_xlmr_results,
        on=RESPONDENT_ID_COLUMN,
        how='left'
    )
)


representative_behavior_records = []


for behavioral_profile in BEHAVIOR_ORDER:

    subset = (
        behavior_source[
            behavior_source[
                'Behavioral_Profile'
            ]
            == behavioral_profile
        ]
        .copy()
    )

    if subset.empty:
        continue

    confident_subset = (
        subset[
            subset[
                'Behavioral_Confidence'
            ]
            .notna()
        ]
    )

    if not confident_subset.empty:

        selected = (
            confident_subset.loc[
                confident_subset[
                    'Behavioral_Confidence'
                ]
                .idxmax()
            ]
        )

    else:

        selected = (
            subset.iloc[0]
        )


    representative_behavior_records.append({

        RESPONDENT_ID_COLUMN:
            selected[
                RESPONDENT_ID_COLUMN
            ],

        'Q7_Response':
            selected[
                risk_preference_col
            ],

        'Q7_Profile':
            selected[
                'Q7_Risk_Profile'
            ],

        'Q9_Response':
            selected[
                loss_reaction_col
            ],

        'Q9_Profile':
            selected[
                'Q9_Loss_Profile'
            ],

        'Final_Behavioral_Profile':
            selected[
                'Behavioral_Profile'
            ],

        'Confidence':
            selected[
                'Behavioral_Confidence'
            ],

        'Decision_Basis':
            selected[
                'Behavioral_Decision_Basis'
            ]
    })


paper_behavior_representatives = (
    pd.DataFrame(
        representative_behavior_records
    )
)


print(
    "\n=== REPRESENTATIVE BEHAVIORAL CLASSIFICATIONS ==="
)

display(
    paper_behavior_representatives
)


# DISTRIBUTION FIGURE

plt.figure(
    figsize=(8, 5)
)

paper_behavior_distribution[
    'Respondents'
].plot(
    kind='bar'
)

plt.title(
    'Distribution of Behavioral Risk Profiles'
)

plt.xlabel(
    'Behavioral Profile'
)

plt.ylabel(
    'Number of Respondents'
)

plt.xticks(
    rotation=0
)

plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        'figure_behavioral_profile_distribution.png'
    ),
    dpi=300,
    bbox_inches='tight'
)


plt.show()


paper_behavior_distribution.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_behavior_distribution.csv'
    )
)


paper_behavior_crosstab.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_behavior_q7_q9_crosstab.csv'
    )
)


paper_behavior_representatives.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_behavior_representatives.csv'
    ),
    index=False
)


This code cell summarizes the behavioral classification results. It calculates the distribution of final behavioral profiles and a cross-tabulation of Q7 and Q9 classifications. It also identifies representative respondents for each behavioral profile. A bar plot of the behavioral profile distribution is generated and saved, along with all summary tables, for reporting.

#  3.3.4 QUANTITATIVE XLM-R BEHAVIORAL SUMMARY


In [ ]:
print(
    "\n=== QUANTITATIVE XLM-R BEHAVIORAL SUMMARY ==="
)

# Keep only respondents with a valid final behavioral profile
behavior_quant = (
    df_xlmr_results[
        df_xlmr_results['Behavioral_Profile'].notna()
    ]
    .copy()
)


# ============================================================
# FINAL BEHAVIORAL PROFILE DISTRIBUTION
# ============================================================

profile_summary = (
    behavior_quant[
        'Behavioral_Profile'
    ]
    .value_counts()
    .rename_axis(
        'Behavioral_Profile'
    )
    .reset_index(
        name='Respondents'
    )
)

profile_summary[
    'Percentage'
] = (
    profile_summary[
        'Respondents'
    ]
    /
    len(behavior_quant)
    * 100
).round(2)


print(
    "\n--- Behavioral Profile Distribution ---"
)

display(
    profile_summary
)


# ============================================================
# FINAL BEHAVIORAL CONFIDENCE SUMMARY
# ============================================================
#
# If Q7 and Q9 agree:
#     confidence = mean of their winning scores
#
# If Neutral is resolved by a directional Q7/Q9:
#     confidence = winning score of the directional response
#
# Direct Low-vs-High conflicts intentionally have no
# numerical final confidence because Neutral was produced
# by the conflict-resolution rule rather than by XLM-R.
# ============================================================

confidence_summary = (
    behavior_quant
    .groupby(
        'Behavioral_Profile'
    )
    .agg(
        Respondents=(
            RESPONDENT_ID_COLUMN,
            'count'
        ),

        Confidence_Available=(
            'Behavioral_Confidence',
            'count'
        ),

        Mean_Confidence=(
            'Behavioral_Confidence',
            'mean'
        ),

        Median_Confidence=(
            'Behavioral_Confidence',
            'median'
        ),

        Std_Confidence=(
            'Behavioral_Confidence',
            'std'
        ),

        Minimum_Confidence=(
            'Behavioral_Confidence',
            'min'
        ),

        Maximum_Confidence=(
            'Behavioral_Confidence',
            'max'
        )
    )
    .round(4)
    .reset_index()
)


print(
    "\n--- Behavioral Confidence Statistics ---"
)

display(
    confidence_summary
)


# ============================================================
# DIAGNOSTIC XLM-R SCORE SUMMARY
# ============================================================


score_summary = (
    behavior_quant
    .groupby(
        'Behavioral_Profile'
    )
    [
        [
            'Risk_Score_Low',
            'Risk_Score_Neutral',
            'Risk_Score_High'
        ]
    ]
    .mean()
    .round(4)
    .reset_index()
)


print(
    "\n--- Mean Diagnostic XLM-R Scores "
    "by Final Behavioral Profile ---"
)

display(
    score_summary
)


# ============================================================
# DECISION-BASIS DISTRIBUTION
# ============================================================

decision_basis_summary = (
    behavior_quant[
        'Behavioral_Decision_Basis'
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        'Decision_Basis'
    )
    .reset_index(
        name='Respondents'
    )
)

decision_basis_summary[
    'Percentage'
] = (
    decision_basis_summary[
        'Respondents'
    ]
    /
    len(behavior_quant)
    * 100
).round(2)


print(
    "\n--- Behavioral Decision Basis ---"
)

display(
    decision_basis_summary
)


# ============================================================
# SAVE TABLES
# ============================================================

profile_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_xlmr_behavior_distribution_quantified.csv'
    ),
    index=False
)

confidence_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_xlmr_confidence_summary.csv'
    ),
    index=False
)

score_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_xlmr_score_summary.csv'
    ),
    index=False
)

decision_basis_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_xlmr_decision_basis_summary.csv'
    ),
    index=False
)

print(
    "\nQuantitative XLM-R behavioral summaries saved."
)

behavioral_candidate_pool = (
    df_xlmr[
        [
            RESPONDENT_ID_COLUMN,
            risk_preference_col,
            loss_reaction_col
        ]
    ]
    .merge(
        df_xlmr_results,
        on=RESPONDENT_ID_COLUMN,
        how='left'
    )
)


print(
    "\n=== CANDIDATE RESPONDENTS FOR "
    "NUMERICAL XLM-R EXAMPLES ==="
)


# Show the most confident respondents
# under each final behavioral profile.

for profile in [
    'Low Risk Tolerance',
    'Neutral',
    'High Risk Tolerance'
]:

    print(
        f"\n--- {profile} ---"
    )

    subset = (
        behavioral_candidate_pool[
            behavioral_candidate_pool[
                'Behavioral_Profile'
            ] == profile
        ]
        .sort_values(
            by='Behavioral_Confidence',
            ascending=False,
            na_position='last'
        )
    )

    display(
        subset[
            [
                RESPONDENT_ID_COLUMN,
                'Q7_Risk_Profile',
                'Q9_Loss_Profile',
                'Behavioral_Profile',
                'Behavioral_Confidence',
                'Behavioral_Decision_Basis',
                risk_preference_col,
                loss_reaction_col
            ]
        ]
        .head(10)
    )

representative_ids = []


for profile in [
    'Low Risk Tolerance',
    'Neutral',
    'High Risk Tolerance'
]:

    subset = (
        df_xlmr_results[
            (
                df_xlmr_results[
                    'Behavioral_Profile'
                ] == profile
            )
            &
            (
                df_xlmr_results[
                    'Behavioral_Confidence'
                ].notna()
            )
        ]
        .sort_values(
            'Behavioral_Confidence',
            ascending=False
        )
    )

    if not subset.empty:

        selected_id = (
            subset.iloc[0][
                RESPONDENT_ID_COLUMN
            ]
        )

        representative_ids.append(
            selected_id
        )


print(
    "Automatically selected respondent IDs:",
    representative_ids
)

SELECTED_RESPONDENTS = [33, 45, 68]


# IMPORTANT:
# This must start as an EMPTY list.
# The dictionaries for Q7 and Q9 will be appended below.
numerical_behavior_examples = []


for respondent_id in SELECTED_RESPONDENTS:

    # --------------------------------------------------------
    # GET ORIGINAL RESPONSES
    # --------------------------------------------------------

    source_row = (
        df_xlmr.loc[
            df_xlmr[
                RESPONDENT_ID_COLUMN
            ] == respondent_id
        ]
    )


    # --------------------------------------------------------
    # GET FINAL EXISTING CLASSIFICATION
    # --------------------------------------------------------

    final_row = (
        df_xlmr_results.loc[
            df_xlmr_results[
                RESPONDENT_ID_COLUMN
            ] == respondent_id
        ]
    )


    if source_row.empty or final_row.empty:

        print(
            f"Skipping Respondent {respondent_id}: "
            "respondent not found."
        )

        continue


    source_row = source_row.iloc[0]
    final_row = final_row.iloc[0]


    q7_text = (
        source_row[
            risk_preference_col
        ]
    )

    q9_text = (
        source_row[
            loss_reaction_col
        ]
    )


    # --------------------------------------------------------
    # RE-RUN THE SAME ITEM-LEVEL CLASSIFICATION
    # TO RETRIEVE ALL THREE NUMERICAL SCORES
    # --------------------------------------------------------

    q7_result = classify_risk_question(
        q7_text,
        q7_labels,
        "The respondent {}."
    )


    q9_result = classify_risk_question(
        q9_text,
        q9_labels,
        (
            "After a large investment loss, "
            "the respondent would choose {}."
        )
    )


    # --------------------------------------------------------
    # QUESTION 7 RECORD
    # --------------------------------------------------------

    if q7_result is not None:

        numerical_behavior_examples.append({

            'Respondent_ID':
                respondent_id,

            'Question':
                'Q7 - Risk Preference',

            'Response':
                q7_text,

            'Low_Score':
                q7_result[
                    'scores'
                ][
                    'Low Risk Tolerance'
                ],

            'Neutral_Score':
                q7_result[
                    'scores'
                ][
                    'Neutral'
                ],

            'High_Score':
                q7_result[
                    'scores'
                ][
                    'High Risk Tolerance'
                ],

            'Item_Classification':
                q7_result[
                    'profile'
                ],

            'Final_Behavioral_Profile':
                final_row[
                    'Behavioral_Profile'
                ],

            'Decision_Basis':
                final_row[
                    'Behavioral_Decision_Basis'
                ]
        })


    # --------------------------------------------------------
    # QUESTION 9 RECORD
    # --------------------------------------------------------

    if q9_result is not None:

        numerical_behavior_examples.append({

            'Respondent_ID':
                respondent_id,

            'Question':
                'Q9 - Loss Reaction',

            'Response':
                q9_text,

            'Low_Score':
                q9_result[
                    'scores'
                ][
                    'Low Risk Tolerance'
                ],

            'Neutral_Score':
                q9_result[
                    'scores'
                ][
                    'Neutral'
                ],

            'High_Score':
                q9_result[
                    'scores'
                ][
                    'High Risk Tolerance'
                ],

            'Item_Classification':
                q9_result[
                    'profile'
                ],

            'Final_Behavioral_Profile':
                final_row[
                    'Behavioral_Profile'
                ],

            'Decision_Basis':
                final_row[
                    'Behavioral_Decision_Basis'
                ]
        })


# ============================================================
# CREATE TABLE
# ============================================================

paper_xlmr_numerical_examples = pd.DataFrame(
    numerical_behavior_examples
)


# Safety check
if paper_xlmr_numerical_examples.empty:

    raise ValueError(
        "No valid behavioral examples were generated. "
        "Check the selected Respondent_ID values."
    )


# ============================================================
# ROUND SCORES FOR PAPER READABILITY
# ============================================================

score_columns = [
    'Low_Score',
    'Neutral_Score',
    'High_Score'
]


for col in score_columns:

    paper_xlmr_numerical_examples[
        col
    ] = (
        paper_xlmr_numerical_examples[
            col
        ]
        .round(4)
    )


# ============================================================
# DISPLAY PAPER-READY TABLE
# ============================================================

print(
    "\n=== TABLE 4.5-B: "
    "EXAMPLE NUMERICAL XLM-R CLASSIFICATIONS ==="
)


display(
    paper_xlmr_numerical_examples[
        [
            'Respondent_ID',
            'Question',
            'Response',

            'Low_Score',
            'Neutral_Score',
            'High_Score',

            'Item_Classification',
            'Final_Behavioral_Profile',
            'Decision_Basis'
        ]
    ]
)


# ============================================================
# SAVE TABLE
# ============================================================

paper_xlmr_numerical_examples.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_table_4_5_b_xlmr_numerical_examples.csv'
    ),
    index=False
)


print(
    "\nTable 4.5-B saved successfully."
)


## 3.4 DATA COMPILATION ACROSS 3 MODULES


In [ ]:
# Merge Financial Capacity (FCM) and Behavioral Profile (XLM-R)
df_compiled = pd.merge(df_fcm_final_output, df_xlmr_results, on=RESPONDENT_ID_COLUMN, how='inner')

# Add the Market Signal (Prophet) to every row
df_compiled['Market_Signal'] = final_market_signal

missing_behavior = (
    df_compiled['Behavioral_Profile']
    .isna()
)

if missing_behavior.any():

    print(
        "\nRespondents excluded from final integration "
        "because no behavioral profile could be determined:"
    )

    display(
        df_compiled.loc[
            missing_behavior,
            [RESPONDENT_ID_COLUMN]
        ]
    )

    df_compiled = (
        df_compiled.loc[
            ~missing_behavior
        ]
        .copy()
    )

print("\nFinal Compiled Data for Integration Module:")
display(df_compiled[[RESPONDENT_ID_COLUMN, 'Financial_Capacity', 'Behavioral_Profile', 'Market_Signal']].head())


This code cell merges the outputs from the Financial Capacity (FCM) and Behavioral Profile (XLM-R) modules into a single DataFrame, `df_compiled`, using the `RESPONDENT_ID_COLUMN`. It then adds the `Market_Signal` from the Prophet module to every row and excludes any respondents for whom a behavioral profile could not be determined, preparing the data for the final integration module.

### 3.4.1 INTEGRATION SUMMARY


In [ ]:
# INTEGRATION SUMMARY

paper_pipeline_counts = pd.DataFrame({

    'Pipeline Stage': [
        'Validated survey respondents',
        'Respondents classified by FCM',
        'Respondents with valid behavioral profile',
        'Respondents retained for final integration'
    ],

    'Respondents': [
        len(df_working),
        len(df_fcm_final_output),
        int(
            df_xlmr_results[
                'Behavioral_Profile'
            ]
            .notna()
            .sum()
        ),
        len(df_compiled)
    ]
})


print(
    "\n=== FINAL PIPELINE RESPONDENT FLOW ==="
)

display(
    paper_pipeline_counts
)


paper_capacity_behavior_crosstab = (
    pd.crosstab(
        df_compiled[
            'Financial_Capacity'
        ],
        df_compiled[
            'Behavioral_Profile'
        ],
        margins=True
    )
)


print(
    "\n=== FINANCIAL CAPACITY × BEHAVIORAL PROFILE ==="
)

display(
    paper_capacity_behavior_crosstab
)


print(
    "\nCurrent Market Signal:",
    final_market_signal
)


paper_pipeline_counts.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_pipeline_counts.csv'
    ),
    index=False
)


paper_capacity_behavior_crosstab.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_capacity_behavior_crosstab.csv'
    )
)


This code cell provides an integration summary, detailing the flow of respondents through the entire pipeline. It presents a table showing the number of respondents at each stage (validated, FCM classified, valid behavioral profile, and final integrated). It also generates a cross-tabulation of Financial Capacity by Behavioral Profile and displays the current Market Signal, saving these tables for reporting.


In [ ]:
print("=== FULL FCM DISTRIBUTION ===")
print(
    df_fcm_final_output[
        'Financial_Capacity'
    ].value_counts()
)

print("\nPercentages:")
print(
    df_fcm_final_output[
        'Financial_Capacity'
    ].value_counts(normalize=True)
    .mul(100)
    .round(2)
)


print("\n=== FULL BEHAVIORAL PROFILE DISTRIBUTION ===")
print(
    df_xlmr_results[
        'Behavioral_Profile'
    ].value_counts(dropna=False)
)

print("\nPercentages:")
print(
    df_xlmr_results[
        'Behavioral_Profile'
    ].value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)


print("\n=== MARKET SIGNAL ===")
print(final_market_signal)


This code cell prints the full distribution of Financial Capacity and Behavioral Profiles, including percentages, from the `df_fcm_final_output` and `df_xlmr_results` DataFrames. It also displays the `final_market_signal`, providing a complete overview of the outputs from the core analytical modules.


In [ ]:
# EXPORT XLM-R BEHAVIORAL CLASSIFICATION FOR REVIEW (FOR TESTING)

from google.colab import files

# Keep the original Q7 and Q9 responses beside the model results
behavioral_review = df_xlmr[
    [
        RESPONDENT_ID_COLUMN,
        risk_preference_col,
        loss_reaction_col
    ]
].copy()

# Rename the long survey questions for easier CSV inspection
behavioral_review = behavioral_review.rename(
    columns={
        risk_preference_col: 'Q7_Risk_Preference_Response',
        loss_reaction_col: 'Q9_Loss_Reaction_Response'
    }
)

# Merge with XLM-R classification results
behavioral_review = behavioral_review.merge(
    df_xlmr_results,
    on=RESPONDENT_ID_COLUMN,
    how='left'
)

# Reorder the useful columns
behavioral_review = behavioral_review[
    [
        RESPONDENT_ID_COLUMN,

        'Q7_Risk_Preference_Response',
        'Q7_Risk_Profile',

        'Q9_Loss_Reaction_Response',
        'Q9_Loss_Profile',

        'Behavioral_Profile',
        'Behavioral_Confidence',

        'Risk_Score_Low',
        'Risk_Score_Neutral',
        'Risk_Score_High'
    ]
]

# Save to CSV
behavioral_review.to_csv(
    'xlmr_behavioral_review.csv',
    index=False
)

print(
    "Behavioral review CSV created successfully."
)

display(
    behavioral_review.head(10)
)

# Download from Google Colab
files.download(
    'xlmr_behavioral_review.csv'
)


This code cell exports the XLM-RoBERTa behavioral classification results for external review. It combines the original Q7 and Q9 responses with the model's classifications, renames columns for clarity, and saves the `behavioral_review` DataFrame to a CSV file. It then provides a download link for the user to retrieve this file, which aids in debugging and validation.


In [ ]:
print("=== CLUSTER SIZE BEFORE LABELING ===")
print(
    df_fcm_results[
        'Cluster_ID'
    ].value_counts()
    .sort_index()
)

print("\n=== CLUSTER -> FINANCIAL CAPACITY MAPPING ===")
for cluster_id, label in cluster_labels.items():
    print(
        f"Cluster {cluster_id} -> {label}"
    )

print("\n=== RAW FINANCIAL CHARACTERISTICS ===")

display(
    fcm_interpretation_data
    .groupby('Financial_Capacity')[
        [
            'Number of Dependents',
            '1. Average Monthly Gross Income',
            '2. Average Monthly Expenses',
            '3. Average Monthly Net Savings'
        ]
    ]
    .mean()
)

print("\n=== FINAL CAPACITY DISTRIBUTION ===")

print(
    df_fcm_final_output[
        'Financial_Capacity'
    ].value_counts()
)


This code cell displays the cluster sizes before labeling, the mapping from numerical cluster IDs to semantic financial capacity labels, the raw mean financial characteristics for each labeled capacity, and the final distribution of financial capacities. This helps in understanding how the FCM module organized respondents based on their financial attributes.

# 4.0. RULE-BASED FEASIBILITY ASSESSMENT MODULE


This module integrates:

1. Behavioral Risk Profile from XLM-RoBERTa
2. Financial Capacity from Fuzzy C-Means
3. Market Signal from Prophet

The three inputs are evaluated jointly through a complete
3 × 3 × 4 rule matrix.

Behavioral Profiles:
    Low Risk Tolerance
    Neutral
    High Risk Tolerance

Financial Capacity:
    Low Financial Capacity
    Medium Financial Capacity
    High Financial Capacity

Market Signals:
    Negative
    Neutral
    Positive
    Volatile

Total rules:
    3 × 3 × 4 = 36

The original six documented rules are preserved exactly.
The remaining combinations formally extend the rule base
using consistent monotonic decision principles.

Baseline Feasibility Score = 0.50
Final Score = Baseline + Rule Delta


In [ ]:
BASELINE_FEASIBILITY = 0.50


# CATEGORY ORDER

BEHAVIOR_LEVELS = [
    'Low Risk Tolerance',
    'Neutral',
    'High Risk Tolerance'
]

CAPACITY_LEVELS = [
    'Low Financial Capacity',
    'Medium Financial Capacity',
    'High Financial Capacity'
]

MARKET_LEVELS = [
    'Negative',
    'Neutral',
    'Positive',
    'Volatile'
]


# DELTA MATRIX

# Capacity values in each list follow:

# [Low Capacity, Medium Capacity, High Capacity]

# Low Risk + Any Capacity + Volatile
#     -> -0.40

# High Risk + High Capacity + Positive
#     -> +0.40

# Neutral + Medium Capacity + Neutral
#     -> 0.00

# High Risk + Low Capacity + Positive
#     -> +0.10

# High Risk + High Capacity + Neutral
#     -> +0.15

# Low Risk + High Capacity + Positive
#     -> +0.10


RULE_DELTA_MATRIX = {


    # NEGATIVE MARKET

    'Negative': {

        'Low Risk Tolerance': [
            -0.40,
            -0.35,
            -0.30
        ],

        'Neutral': [
            -0.30,
            -0.25,
            -0.20
        ],

        'High Risk Tolerance': [
            -0.20,
            -0.10,
            -0.05
        ]
    },


    # NEUTRAL / FLAT MARKET

    'Neutral': {

        'Low Risk Tolerance': [
            -0.15,
            -0.10,
            -0.05
        ],

        'Neutral': [
            -0.10,
             0.00,
             0.05
        ],

        'High Risk Tolerance': [
            -0.05,
             0.05,
             0.15
        ]
    },


    # POSITIVE MARKET

    'Positive': {

        'Low Risk Tolerance': [
             0.00,
             0.05,
             0.10
        ],

        'Neutral': [
             0.05,
             0.15,
             0.25
        ],

        'High Risk Tolerance': [
             0.10,
             0.25,
             0.40
        ]
    },


    # VOLATILE / HIGH-UNCERTAINTY MARKET

    # Low Risk users receive a -0.40 adjustment regardless of capacity.
    # Neutral and High Risk users are progressively less restricted as financial capacity increases.

    'Volatile': {

        'Low Risk Tolerance': [
            -0.40,
            -0.40,
            -0.40
        ],

        'Neutral': [
            -0.30,
            -0.20,
            -0.10
        ],

        'High Risk Tolerance': [
            -0.15,
             0.00,
             0.10
        ]
    }
}


# RULE OVERRIDES


ORIGINAL_RULE_OVERRIDES = {

    (
        'Low Risk Tolerance',
        'Low Financial Capacity',
        'Volatile'
    ): (
        'Strongly decrease',
        'Conservative advice'
    ),

    (
        'Low Risk Tolerance',
        'Medium Financial Capacity',
        'Volatile'
    ): (
        'Strongly decrease',
        'Conservative advice'
    ),

    (
        'Low Risk Tolerance',
        'High Financial Capacity',
        'Volatile'
    ): (
        'Strongly decrease',
        'Conservative advice'
    ),


    (
        'High Risk Tolerance',
        'High Financial Capacity',
        'Positive'
    ): (
        'Strongly increase',
        'Aggressive advice'
    ),


    (
        'Neutral',
        'Medium Financial Capacity',
        'Neutral'
    ): (
        'Maintain baseline',
        'Balanced advice'
    ),


    (
        'High Risk Tolerance',
        'Low Financial Capacity',
        'Positive'
    ): (
        'Restricted increase',
        'Cautious advice'
    ),


    (
        'High Risk Tolerance',
        'High Financial Capacity',
        'Neutral'
    ): (
        'Slight increase',
        'Accumulation advice'
    ),


    (
        'Low Risk Tolerance',
        'High Financial Capacity',
        'Positive'
    ): (
        'Slight increase',
        'Conservative growth advice'
    )
}


# DESCRIBE EXTENDED RULES

def describe_extended_rule(
    behavior,
    capacity,
    market,
    delta
):

    key = (
        behavior,
        capacity,
        market
    )


    # Preserve original documented rules

    if key in ORIGINAL_RULE_OVERRIDES:

        return (
            ORIGINAL_RULE_OVERRIDES[
                key
            ]
        )


    # Extended rule terminology

    if delta <= -0.35:

        return (
            'Strongly decrease',
            'Conservative advice'
        )


    elif delta <= -0.20:

        return (
            'Decrease',
            'Defensive advice'
        )


    elif delta < 0:

        return (
            'Slight decrease',
            'Cautious preservation advice'
        )


    elif delta == 0:

        return (
            'Maintain baseline',
            'Balanced advice'
        )


    elif delta <= 0.10:

        return (
            'Restricted increase',
            'Cautious growth advice'
        )


    elif delta <= 0.25:

        return (
            'Slight increase',
            'Accumulation advice'
        )


    else:

        return (
            'Strongly increase',
            'Aggressive advice'
        )


# BUILD ALL 36 RULES

system_rules = []


for market in MARKET_LEVELS:

    for behavior in BEHAVIOR_LEVELS:

        deltas = (
            RULE_DELTA_MATRIX[
                market
            ][
                behavior
            ]
        )


        for capacity_index, capacity in enumerate(
            CAPACITY_LEVELS
        ):

            delta = float(
                deltas[
                    capacity_index
                ]
            )


            (
                system_rule,
                rule_result
            ) = describe_extended_rule(
                behavior,
                capacity,
                market,
                delta
            )


            system_rules.append({

                'Behavioral_Profile':
                    behavior,

                'Financial_Capacity':
                    capacity,

                'Market_Signal':
                    market,

                'System_Rule':
                    system_rule,

                'Delta':
                    delta,

                'Rule_Result':
                    rule_result
            })


# VERIFY RULE TABLE

df_rule_table = pd.DataFrame(
    system_rules
)


print(
    "\nTotal system rules generated:",
    len(df_rule_table)
)


if len(df_rule_table) != 36:

    raise ValueError(
        "The feasibility rule table "
        "must contain exactly 36 rules."
    )


duplicate_rules = (
    df_rule_table.duplicated(
        subset=[
            'Behavioral_Profile',
            'Financial_Capacity',
            'Market_Signal'
        ]
    )
    .sum()
)


if duplicate_rules > 0:

    raise ValueError(
        f"Duplicate feasibility rules found: "
        f"{duplicate_rules}"
    )


print(
    "Rule matrix validation successful."
)


print(
    "\n--- COMPLETE 36-RULE MATRIX ---"
)

display(
    df_rule_table
)


# FIND RULE FOR A RESPONDENT

def find_system_rule(
    behavioral_profile,
    financial_capacity,
    market_signal
):

    matches = df_rule_table[
        (
            df_rule_table[
                'Behavioral_Profile'
            ]
            == behavioral_profile
        )
        &
        (
            df_rule_table[
                'Financial_Capacity'
            ]
            == financial_capacity
        )
        &
        (
            df_rule_table[
                'Market_Signal'
            ]
            == market_signal
        )
    ]


    if len(matches) == 1:

        return (
            matches.iloc[0]
            .to_dict()
        )


    return None


# CALCULATE RULE-BASED FEASIBILITY

def calculate_rule_based_feasibility(
    row
):

    behavioral_profile = (
        row[
            'Behavioral_Profile'
        ]
    )

    financial_capacity = (
        row[
            'Financial_Capacity'
        ]
    )

    market_signal = (
        row[
            'Market_Signal'
        ]
    )


    # Missing module output

    if pd.isna(
        behavioral_profile
    ):

        return pd.Series([
            np.nan,
            np.nan,
            'No behavioral profile',
            'No recommendation',
            (
                "Behavioral profile "
                "could not be determined."
            )
        ])


    # Find exact three-variable rule

    matched_rule = (
        find_system_rule(
            behavioral_profile,
            financial_capacity,
            market_signal
        )
    )


    if matched_rule is None:

        explanation = (
            "No matching feasibility "
            "rule was found for: "
            f"{behavioral_profile} + "
            f"{financial_capacity} + "
            f"{market_signal}."
        )

        return pd.Series([
            np.nan,
            np.nan,
            'Rule lookup error',
            'No recommendation',
            explanation
        ])


    # Apply rule delta

    delta = (
        matched_rule[
            'Delta'
        ]
    )


    score = (
        BASELINE_FEASIBILITY
        + delta
    )


    # Keep score within 0–1
    score = max(
        0.0,
        min(
            1.0,
            score
        )
    )


    explanation = (

        f"Behavior: "
        f"{behavioral_profile} | "

        f"Capacity: "
        f"{financial_capacity} | "

        f"Market: "
        f"{market_signal} | "

        f"Rule: "
        f"{matched_rule['System_Rule']} | "

        f"Delta: "
        f"{delta:+.2f}"
    )


    return pd.Series([
        score,
        delta,
        matched_rule[
            'System_Rule'
        ],
        matched_rule[
            'Rule_Result'
        ],
        explanation
    ])


# APPLY ALL RULES TO COMPILED RESPONDENTS

print(
    "\nRunning Complete "
    "Rule-Based Feasibility Assessment..."
)


df_compiled[
    [
        'Feasibility_Score',
        'Rule_Delta',
        'System_Rule',
        'Rule_Result',
        'System_Reasoning'
    ]
] = (
    df_compiled.apply(
        calculate_rule_based_feasibility,
        axis=1
    )
)


# REPORT

print(
    "\n--- FEASIBILITY ASSESSMENT RESULTS ---"
)


display(
    df_compiled[
        [
            RESPONDENT_ID_COLUMN,

            'Financial_Capacity',
            'Behavioral_Profile',
            'Market_Signal',

            'Rule_Delta',
            'Feasibility_Score',

            'System_Rule',
            'Rule_Result'
        ]
    ]
    .head(20)
)


# CHECK THAT EVERY RESPONDENT RECEIVED A RULE

missing_rules = (
    df_compiled[
        'Feasibility_Score'
    ]
    .isna()
)


print(
    "\nRespondents without a valid "
    "feasibility rule:",
    missing_rules.sum()
)


if missing_rules.any():

    display(
        df_compiled.loc[
            missing_rules,
            [
                RESPONDENT_ID_COLUMN,
                'Financial_Capacity',
                'Behavioral_Profile',
                'Market_Signal'
            ]
        ]
    )


# RULE USAGE DISTRIBUTION

print(
    "\nSystem Rule Distribution:"
)

print(
    df_compiled[
        'System_Rule'
    ]
    .value_counts()
)


print(
    "\nRule Result Distribution:"
)

print(
    df_compiled[
        'Rule_Result'
    ]
    .value_counts()
)


print(
    "\nFeasibility Score Summary:"
)

display(
    df_compiled[
        'Feasibility_Score'
    ]
    .describe()
)


This module integrates the Behavioral Risk Profile, Financial Capacity, and Market Signal using a 3x3x4 rule matrix to determine a 'Feasibility_Score' and an investment recommendation. It defines category levels, a `RULE_DELTA_MATRIX` for score adjustments, and `ORIGINAL_RULE_OVERRIDES` for specific cases. It then builds a comprehensive rule table, validates it, and applies these rules to each respondent in `df_compiled` to generate feasibility scores, system rules, and reasoning.

### 4.1. Complete Rule-Matrix Output
The feasibility matrix contains all 36 combinations produced by three Behavioral Profiles, three Financial Capacity categories, and four Market Signals. The Rule Delta values are researcher-defined heuristic decision rules rather than model-learned parameters.

The table below presents the corresponding Rule Delta, final Feasibility Score, System Rule, and Rule Result for every possible combination.


In [ ]:
# COMPLETE 36-RULE PAPER OUTPUT

paper_rule_table = (
    df_rule_table
    .copy()
)


paper_rule_table[
    'Feasibility_Score'
] = (
    BASELINE_FEASIBILITY
    +
    paper_rule_table[
        'Delta'
    ]
).clip(
    lower=0.0,
    upper=1.0
)


paper_rule_table = (
    paper_rule_table[
        [
            'Market_Signal',
            'Behavioral_Profile',
            'Financial_Capacity',
            'Delta',
            'Feasibility_Score',
            'System_Rule',
            'Rule_Result'
        ]
    ]
)


print(
    "\n=== COMPLETE 36-RULE FEASIBILITY MATRIX ==="
)

display(
    paper_rule_table
)


# COMPACT MATRIX FOR EACH MARKET CONDITION

for market_condition in MARKET_LEVELS:

    market_rules = (
        paper_rule_table[
            paper_rule_table[
                'Market_Signal'
            ]
            == market_condition
        ]
        .copy()
    )


    market_rules[
        'Paper_Cell'
    ] = (
        market_rules.apply(
            lambda row:
                (
                    f"Δ {row['Delta']:+.2f} | "
                    f"Score {row['Feasibility_Score']:.2f} | "
                    f"{row['Rule_Result']}"
                ),
            axis=1
        )
    )


    matrix = (
        market_rules.pivot(
            index='Behavioral_Profile',
            columns='Financial_Capacity',
            values='Paper_Cell'
        )
        .reindex(
            index=BEHAVIOR_LEVELS,
            columns=CAPACITY_LEVELS
        )
    )


    print(
        f"\n=== {market_condition.upper()} MARKET MATRIX ==="
    )

    display(
        matrix
    )


paper_rule_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'complete_36_rule_matrix.csv'
    ),
    index=False
)


This code cell generates and displays the complete 36-rule feasibility matrix. It calculates the final `Feasibility_Score` for each rule combination by adding the `Rule_Delta` to a `BASELINE_FEASIBILITY` score. It then presents compact matrices for each market condition (Negative, Neutral, Positive, Volatile) and saves the complete rule table to a CSV file.

# 5.0 INVESTMENT ASSESSMENT AND RECOMMENDATION MODULE


Section 4 already determines the respondent's feasibility score,
System Rule, and Rule Result using:

    Financial Capacity
    + Behavioral Risk Profile
    + Market Signal

This prevents a second decision layer from overriding or
contradicting the complete 36-rule feasibility matrix.

This prevents a second decision layer from overriding or
contradicting the complete 36-rule feasibility matrix.


In [ ]:
# USER-FACING RECOMMENDATION

RECOMMENDATION_TEMPLATES = {


    # CONSERVATIVE

    "Conservative advice": {

        "assessment":
            "Conservative Position",

        "advice":
            (
                "Prioritize financial stability and capital preservation. "
                "Avoid taking substantial investment risk under the current "
                "combination of financial capacity, risk tolerance, and "
                "market conditions. Maintain sufficient liquid savings and "
                "only consider investment exposure when personal finances "
                "and market conditions are more favorable."
            )
    },


    # DEFENSIVE

    "Defensive advice": {

        "assessment":
            "Defensive Position",

        "advice":
            (
                "Maintain a cautious financial position. Focus on liquidity, "
                "financial reserves, and lower-risk options rather than "
                "significantly increasing investment exposure. Reassess "
                "when market conditions or personal financial capacity improve."
            )
    },


    # CAUTIOUS PRESERVATION

    "Cautious preservation advice": {

        "assessment":
            "Cautious Preservation",

        "advice":
            (
                "Preserve most available capital while allowing only limited "
                "investment exposure if appropriate. Avoid committing funds "
                "needed for essential expenses or short-term financial needs."
            )
    },


    # BALANCED

    "Balanced advice": {

        "assessment":
            "Balanced Position",

        "advice":
            (
                "Maintain a balanced approach between financial safety and "
                "investment growth. Consider diversified investments using "
                "only disposable funds while maintaining adequate savings "
                "and liquidity."
            )
    },


    # CAUTIOUS ADVICE

    "Cautious advice": {

        "assessment":
            "Cautious Growth",

        "advice":
            (
                "The current conditions allow some investment exposure, "
                "but financial capacity places an important limit on how "
                "aggressively funds should be committed. Prefer gradual, "
                "diversified investment using only disposable funds."
            )
    },


    # CAUTIOUS GROWTH

    "Cautious growth advice": {

        "assessment":
            "Cautious Growth",

        "advice":
            (
                "A limited increase in investment exposure may be reasonable, "
                "but financial stability should remain the priority. Consider "
                "gradual and diversified investing rather than concentrating "
                "funds in highly speculative assets."
            )
    },


    # CONSERVATIVE GROWTH

    "Conservative growth advice": {

        "assessment":
            "Conservative Growth",

        "advice":
            (
                "Financial capacity may support limited growth-oriented "
                "investment, but the respondent's low risk tolerance should "
                "remain the primary constraint. Favor diversified and "
                "lower-volatility investments rather than aggressive exposure."
            )
    },


    # ACCUMULATION

    "Accumulation advice": {

        "assessment":
            "Gradual Accumulation",

        "advice":
            (
                "The respondent's profile supports gradually increasing "
                "investment exposure. Consider diversified investments and "
                "incremental allocation while retaining sufficient liquid "
                "funds for financial obligations and emergencies."
            )
    },


    # AGGRESSIVE

    "Aggressive advice": {

        "assessment":
            "Growth-Oriented Position",

        "advice":
            (
                "The combined financial, behavioral, and market profile "
                "supports a more growth-oriented investment approach. "
                "Higher-risk investments may be considered using disposable "
                "investment funds, preferably through diversification rather "
                "than concentrated exposure."
            )
    }
}


# VERIFY THAT EVERY RULE RESULT HAS A TEMPLATE

rule_results_in_matrix = set(
    df_rule_table[
        'Rule_Result'
    ]
    .unique()
)


recommendation_template_keys = set(
    RECOMMENDATION_TEMPLATES.keys()
)


missing_templates = (
    rule_results_in_matrix
    -
    recommendation_template_keys
)


if missing_templates:

    raise ValueError(
        "Missing recommendation templates for: "
        f"{sorted(missing_templates)}"
    )


print(
    "Recommendation-template validation successful. "
    "Every rule result has a corresponding recommendation."
)


# GENERATE FINAL RECOMMENDATION

def generate_final_recommendation(row):

    rule_result = (
        row[
            'Rule_Result'
        ]
    )

    system_rule = (
        row[
            'System_Rule'
        ]
    )

    feasibility_score = (
        row[
            'Feasibility_Score'
        ]
    )


    if (
        pd.isna(rule_result)
        or
        rule_result not in RECOMMENDATION_TEMPLATES
    ):

        return pd.Series([
            "Assessment unavailable",

            (
                "A recommendation could not be generated because "
                "a valid feasibility rule was not available."
            )
        ])


    template = (
        RECOMMENDATION_TEMPLATES[
            rule_result
        ]
    )


    assessment = (
        template[
            'assessment'
        ]
    )


    recommendation = (
        template[
            'advice'
        ]
    )


    recommendation = (
        f"{recommendation} "
        f"[System rule: {system_rule}; "
        f"Feasibility score: {feasibility_score:.2f}]"
    )


    return pd.Series([
        assessment,
        recommendation
    ])


# APPLY RECOMMENDATION MODULE

df_compiled[
    [
        'Investment_Assessment',
        'Recommendation_Advice'
    ]
] = (
    df_compiled.apply(
        generate_final_recommendation,
        axis=1
    )
)


# FINAL RECOMMENDATION REPORT

print(
    "\n=== FINAL INVESTMENT ASSESSMENTS ==="
)


display(
    df_compiled[
        [
            RESPONDENT_ID_COLUMN,
            'Financial_Capacity',
            'Behavioral_Profile',
            'Market_Signal',
            'Feasibility_Score',
            'System_Rule',
            'Rule_Result',
            'Investment_Assessment',
            'Recommendation_Advice'
        ]
    ]
    .head(20)
)


print(
    "\nInvestment Assessment Distribution:"
)


print(
    df_compiled[
        'Investment_Assessment'
    ]
    .value_counts()
)


This code cell defines a set of `RECOMMENDATION_TEMPLATES` that map each `Rule_Result` (e.g., 'Conservative advice', 'Aggressive advice') to a specific `assessment` and detailed `advice`. It verifies that every possible `Rule_Result` has a corresponding template. It then applies this module to `df_compiled` to generate `Investment_Assessment` and `Recommendation_Advice` for each respondent, providing a final user-facing output.

## 5.1 COMPLETE 36-RULE ADVISORY OUTPUT


In [ ]:
# COMPLETE 36-RULE ADVISORY OUTPUT

paper_all_rule_advice = (
    paper_rule_table
    .copy()
)


paper_all_rule_advice[
    'Investment_Assessment'
] = (
    paper_all_rule_advice[
        'Rule_Result'
    ]
    .map(
        lambda result:
            RECOMMENDATION_TEMPLATES[
                result
            ][
                'assessment'
            ]
    )
)


paper_all_rule_advice[
    'Recommendation_Advice'
] = (
    paper_all_rule_advice[
        'Rule_Result'
    ]
    .map(
        lambda result:
            RECOMMENDATION_TEMPLATES[
                result
            ][
                'advice'
            ]
    )
)


print(
    "\n=== ALL 36 POSSIBLE ADVISORY OUTPUTS ==="
)


display(
    paper_all_rule_advice
)


paper_all_rule_advice.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'complete_36_rule_advisory_outputs.csv'
    ),
    index=False
)


This code cell generates and displays the complete advisory output for all 36 possible rule combinations. It maps the `Rule_Result` from the `paper_rule_table` to the corresponding `Investment_Assessment` and `Recommendation_Advice` using the `RECOMMENDATION_TEMPLATES`. The complete advisory outputs are then saved to a CSV file.

## 5.2 SIMULATED MARKET-SCENARIO DEMONSTRATION


In [ ]:
# SIMULATED MARKET-SCENARIO DEMONSTRATION

# This is an ILLUSTRATIVE SENSITIVITY ANALYSIS.
# These hypothetical market values are NOT actual Prophet results.
# They demonstrate how the same respondent would receive different
# advice if the market signal changed.



demo_respondent = (
    df_compiled
    .sample(
        n=1,
        random_state=RANDOM_SEED
    )
    .iloc[0]
)


print(
    "\n=== RANDOMLY SELECTED RESPONDENT "
    "FOR SCENARIO DEMONSTRATION ==="
)


display(
    pd.DataFrame({
        'Respondent_ID': [
            demo_respondent[
                RESPONDENT_ID_COLUMN
            ]
        ],

        'Financial_Capacity': [
            demo_respondent[
                'Financial_Capacity'
            ]
        ],

        'Behavioral_Profile': [
            demo_respondent[
                'Behavioral_Profile'
            ]
        ],

        'Actual_Market_Signal': [
            demo_respondent[
                'Market_Signal'
            ]
        ]
    })
)


# Illustrative values chosen only to trigger
# each defined market-signal category.

simulated_market_conditions = [

    {
        'Scenario':
            'Positive Market',

        'Projected_Growth':
            0.02,

        'Forecast_Uncertainty':
            0.04
    },

    {
        'Scenario':
            'Neutral Market',

        'Projected_Growth':
            0.00,

        'Forecast_Uncertainty':
            0.04
    },

    {
        'Scenario':
            'Negative Market',

        'Projected_Growth':
            -0.02,

        'Forecast_Uncertainty':
            0.04
    },

    {
        'Scenario':
            'Volatile / High-Uncertainty Market',

        'Projected_Growth':
            0.00,

        'Forecast_Uncertainty':
            0.08
    }
]


def classify_simulated_market(
    growth_rate,
    uncertainty_ratio
):

    if (
        uncertainty_ratio
        >
        FORECAST_UNCERTAINTY_THRESHOLD
    ):
        return 'Volatile'

    elif (
        growth_rate
        >=
        POSITIVE_THRESHOLD
    ):
        return 'Positive'

    elif (
        growth_rate
        <=
        NEGATIVE_THRESHOLD
    ):
        return 'Negative'

    else:
        return 'Neutral'


scenario_records = []


for scenario in simulated_market_conditions:

    simulated_signal = (
        classify_simulated_market(
            scenario[
                'Projected_Growth'
            ],
            scenario[
                'Forecast_Uncertainty'
            ]
        )
    )


    matched_rule = (
        find_system_rule(

            demo_respondent[
                'Behavioral_Profile'
            ],

            demo_respondent[
                'Financial_Capacity'
            ],

            simulated_signal
        )
    )


    delta = (
        matched_rule[
            'Delta'
        ]
    )


    simulated_score = max(
        0.0,
        min(
            1.0,
            BASELINE_FEASIBILITY
            +
            delta
        )
    )


    rule_result = (
        matched_rule[
            'Rule_Result'
        ]
    )


    recommendation_template = (
        RECOMMENDATION_TEMPLATES[
            rule_result
        ]
    )


    scenario_records.append({

        'Scenario':
            scenario[
                'Scenario'
            ],

        'Projected_Growth_%':
            scenario[
                'Projected_Growth'
            ]
            * 100,

        'Forecast_Uncertainty_%':
            scenario[
                'Forecast_Uncertainty'
            ]
            * 100,

        'Market_Signal':
            simulated_signal,

        'Financial_Capacity':
            demo_respondent[
                'Financial_Capacity'
            ],

        'Behavioral_Profile':
            demo_respondent[
                'Behavioral_Profile'
            ],

        'Rule_Delta':
            delta,

        'Feasibility_Score':
            simulated_score,

        'System_Rule':
            matched_rule[
                'System_Rule'
            ],

        'Rule_Result':
            rule_result,

        'Investment_Assessment':
            recommendation_template[
                'assessment'
            ],

        'Recommendation':
            recommendation_template[
                'advice'
            ]
    })


paper_market_scenario_demo = (
    pd.DataFrame(
        scenario_records
    )
)


print(
    "\n=== SIMULATED MARKET SCENARIO ANALYSIS ==="
)

print(
    "NOTE: These four market conditions are illustrative "
    "scenario tests, not empirical Prophet forecasts."
)


display(
    paper_market_scenario_demo
)


paper_market_scenario_demo.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_simulated_market_scenarios.csv'
    ),
    index=False
)


This code cell demonstrates how the advisory system would react to different simulated market scenarios for a randomly selected respondent. It defines hypothetical market conditions (Positive, Neutral, Negative, Volatile) and classifies them using a simplified market signal function. For each scenario, it applies the rule-based feasibility assessment to the demo respondent and displays the resulting investment assessment and recommendation, highlighting the system's adaptability.

# 6.0 VALIDATION

## 6.1. Prophet Forecasting Evaluation (RMSE, MAE & MAPE)


In [ ]:
# PROPHET FORECASTING EVALUATION
import os
from prophet.diagnostics import cross_validation, performance_metrics

print("\n=== PROPHET CROSS-VALIDATION ===")

try:
    # Run cross-validation over the historical data
    df_cv = cross_validation(m, initial='730 days', period='180 days', horizon='30 days')
    df_p = performance_metrics(df_cv)

    # Calculate average metrics
    avg_rmse = df_p['rmse'].mean()
    avg_mae = df_p['mae'].mean()
    avg_mape = df_p['mape'].mean() * 100 if 'mape' in df_p.columns else np.nan

    # 3. Format metrics into a clean DataFrame for the thesis
    paper_prophet_metrics = pd.DataFrame({
        'Metric': ['RMSE', 'MAE', 'MAPE (%)'],
        'Value': [avg_rmse, avg_mae, avg_mape]
    }).round(4)

    print("\n=== PROPHET MODEL METRICS ===")
    display(paper_prophet_metrics)

    # PLOT MOST RECENT CROSS-VALIDATION WINDOW
    # We grab the most recent test window to visually compare Actual vs Predicted
    latest_cutoff = df_cv['cutoff'].max()
    latest_cv_window = df_cv[df_cv['cutoff'] == latest_cutoff].sort_values('ds')

    plt.figure(figsize=(10, 5))
    plt.plot(latest_cv_window['ds'], latest_cv_window['y'], label='Actual')
    plt.plot(latest_cv_window['ds'], latest_cv_window['yhat'], label='Predicted')

    plt.title('Prophet Cross-Validation: Actual vs Predicted PSEi')
    plt.xlabel('Date')
    plt.ylabel('PSEi Index Level')
    plt.legend()
    plt.tight_layout()

    # Save the figure and the metrics to your output folder
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    plt.savefig(os.path.join(OUTPUT_DIR, 'figure_prophet_cross_validation.png'), dpi=300, bbox_inches='tight')
    plt.show()

    paper_prophet_metrics.to_csv(os.path.join(OUTPUT_DIR, 'paper_prophet_metrics.csv'), index=False)
    print("Prophet metrics and validation graph saved successfully.")

except Exception as e:
    print(f"Prophet cross-validation failed: {e}")


This code cell evaluates the Prophet forecasting model using cross-validation. It performs cross-validation with an initial training period, forecast period, and horizon, then calculates performance metrics such as RMSE, MAE, and MAPE. These metrics are formatted into a table and displayed. It also plots the most recent cross-validation window to visualize actual vs. predicted values and saves the plot and metrics to CSV.

## 6.2. Fuzzy C-Means Evaluation


In [ ]:
# FUZZY C-MEANS EVALUATION
import os
from sklearn.metrics import silhouette_score

print("\n=== FCM CLUSTERING EVALUATION ===")

# Calculate Silhouette Score using the hard-assigned clusters
sil_score = silhouette_score(df_fcm_scaled, cluster_membership)

# Format metrics into a clean DataFrame for the thesis
paper_fcm_metrics = pd.DataFrame({
    'Metric': [
        'Fuzzy Partition Coefficient (FPC)',
        'Silhouette Score'
    ],
    'Value': [
        fpc,
        sil_score
    ],
    'Interpretation': [
        "Higher values indicate more concentrated and less ambiguous fuzzy memberships.",
        "Higher values indicate stronger geometric separation between dominant cluster assignments."
    ]
})

display(paper_fcm_metrics)

# Save the metrics to your output folder
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

paper_fcm_metrics.to_csv(os.path.join(OUTPUT_DIR, 'paper_fcm_metrics.csv'), index=False)
print("FCM metrics saved successfully.")


This code cell evaluates the Fuzzy C-Means clustering using the Silhouette Score. It calculates the Silhouette Score based on the scaled FCM data and the hard-assigned clusters. The FPC (Fuzzy Partition Coefficient) and Silhouette Score are then presented in a DataFrame along with their interpretations, and this metrics table is saved to a CSV file.

## 6.3. XLM-RoBERTa (Behavioral NLP) Evaluation

### 6.3.1 CREATE RESEARCHER-CONSENSUS VALIDATION SAMPLE


In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

VALIDATION_SAMPLE_SIZE = 25
VALIDATION_RANDOM_SEED = 42


# Build source table containing the actual behavioral answers

researcher_validation_source = df_xlmr[
    [
        RESPONDENT_ID_COLUMN,
        risk_preference_col,
        loss_reaction_col
    ]
].copy()


researcher_validation_source = (
    researcher_validation_source
    .rename(
        columns={
            risk_preference_col:
                'Q7_Risk_Preference',

            loss_reaction_col:
                'Q9_Loss_Reaction'
        }
    )
)


# Keep respondents that contain at least one usable answer

researcher_validation_source = (
    researcher_validation_source[
        researcher_validation_source[
            [
                'Q7_Risk_Preference',
                'Q9_Loss_Reaction'
            ]
        ]
        .notna()
        .any(axis=1)
    ]
    .copy()
)


# Randomly select 25 respondents reproducibly

researcher_validation_sample = (
    researcher_validation_source
    .sample(
        n=min(
            VALIDATION_SAMPLE_SIZE,
            len(researcher_validation_source)
        ),
        random_state=VALIDATION_RANDOM_SEED
    )
    .sort_values(RESPONDENT_ID_COLUMN)
    .reset_index(drop=True)
)


# Create blank researcher-label columns

researcher_validation_sample[
    'Researcher_1_Label'
] = ""

researcher_validation_sample[
    'Researcher_2_Label'
] = ""

researcher_validation_sample[
    'Researcher_3_Label'
] = ""


# Add labeling instructions

print("""
LABELING GUIDE

Low Risk Tolerance:
- Prefers banks, savings, or safer options
- Avoids risky investments
- Would panic, sell, or avoid further investing after a loss

Neutral:
- Prefers a balanced/mixed investment approach
- Would hold, observe, or wait after a loss
- Shows moderate or mixed risk behavior

High Risk Tolerance:
- Explicitly prefers high-risk/high-reward investments
- Is willing to accept substantial losses
- Would buy more or continue taking substantial risk after a decline

Researchers should label each respondent independently.
Do NOT inspect the model prediction before assigning labels.
""")


display(
    researcher_validation_sample
)


# Export annotation sheet

researcher_validation_sample.to_csv(
    'researcher_consensus_annotation_sheet.csv',
    index=False
)

files.download(
    'researcher_consensus_annotation_sheet.csv'
)


This code cell prepares a validation sample for evaluating the XLM-RoBERTa behavioral classification. It selects a reproducible random sample of 25 respondents from `df_xlmr` who have at least one usable open-ended response. It then adds blank columns for researcher labels, provides a labeling guide, displays the sample, and exports it as an annotation sheet CSV for human consensus labeling.

### 6.3.2 XLM-R RESEARCHER-CONSENSUS VALIDATION


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from google.colab import files


print(
    "Upload the completed researcher-consensus "
    "annotation CSV:"
)

uploaded_validation = files.upload()

validation_filename = next(
    iter(uploaded_validation)
)

df_validation = pd.read_csv(
    io.BytesIO(
        uploaded_validation[
            validation_filename
        ]
    )
)


# ALLOWED LABELS

VALID_BEHAVIOR_LABELS = [
    'Low Risk Tolerance',
    'Neutral',
    'High Risk Tolerance'
]


# CLEAN LABELS

researcher_columns = [
    'Researcher_1_Label',
    'Researcher_2_Label',
    'Researcher_3_Label'
]

for col in researcher_columns:

    df_validation[col] = (
        df_validation[col]
        .astype(str)
        .str.strip()
    )

# MAJORITY CONSENSUS


def determine_consensus(row):

    labels = [
        row['Researcher_1_Label'],
        row['Researcher_2_Label'],
        row['Researcher_3_Label']
    ]

    labels = [
        label
        for label in labels
        if label in VALID_BEHAVIOR_LABELS
    ]


    # Fewer than 2 usable researcher labels
    if len(labels) < 2:
        return np.nan


    counts = pd.Series(
        labels
    ).value_counts()


    # At least two researchers agree
    if counts.iloc[0] >= 2:
        return counts.index[0]


    # All three disagree
    return np.nan


df_validation[
    'Researcher_Consensus'
] = (
    df_validation.apply(
        determine_consensus,
        axis=1
    )
)


# DISPLAY CONSENSUS STATUS


print(
    "\nResearcher Consensus Distribution:"
)

print(
    df_validation[
        'Researcher_Consensus'
    ]
    .value_counts(
        dropna=False
    )
)


no_consensus = (
    df_validation[
        'Researcher_Consensus'
    ]
    .isna()
)

if no_consensus.any():

    print(
        "\nProfiles requiring researcher adjudication:"
    )

    display(
        df_validation.loc[
            no_consensus,
            [
                RESPONDENT_ID_COLUMN,
                'Q7_Risk_Preference',
                'Q9_Loss_Reaction',
                'Researcher_1_Label',
                'Researcher_2_Label',
                'Researcher_3_Label'
            ]
        ]
    )


# MERGE CONSENSUS WITH MODEL OUTPUT

df_validation_results = pd.merge(
    df_validation,
    df_xlmr_results[
        [
            RESPONDENT_ID_COLUMN,
            'Behavioral_Profile'
        ]
    ],
    on=RESPONDENT_ID_COLUMN,
    how='left'
)


# Only cases with an established human consensus
df_evaluable = (
    df_validation_results[
        df_validation_results[
            'Researcher_Consensus'
        ]
        .notna()
    ]
    .copy()
)


y_true = (
    df_evaluable[
        'Researcher_Consensus'
    ]
)

y_pred = (
    df_evaluable[
        'Behavioral_Profile'
    ]
)


# METRICS

accuracy = accuracy_score(
    y_true,
    y_pred
)

macro_precision = precision_score(
    y_true,
    y_pred,
    labels=VALID_BEHAVIOR_LABELS,
    average='macro',
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    labels=VALID_BEHAVIOR_LABELS,
    average='macro',
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=VALID_BEHAVIOR_LABELS,
    average='macro',
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=VALID_BEHAVIOR_LABELS,
    average='weighted',
    zero_division=0
)


print(
    "\n=== XLM-R RESEARCHER-CONSENSUS "
    "VALIDATION ==="
)

print(
    f"Evaluated profiles: "
    f"{len(df_evaluable)}"
)

print(
    f"Accuracy: "
    f"{accuracy:.4f}"
)

print(
    f"Macro Precision: "
    f"{macro_precision:.4f}"
)

print(
    f"Macro Recall: "
    f"{macro_recall:.4f}"
)

print(
    f"Macro F1: "
    f"{macro_f1:.4f}"
)

print(
    f"Weighted F1: "
    f"{weighted_f1:.4f}"
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_true,
        y_pred,
        labels=VALID_BEHAVIOR_LABELS,
        zero_division=0
    )
)


# CONFUSION MATRIX

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=VALID_BEHAVIOR_LABELS
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        'Low',
        'Neutral',
        'High'
    ]
)

disp.plot()

plt.title(
    'XLM-R Behavioral Classification\n'
    'vs Researcher Consensus'
)

plt.show()


# SHOW DISAGREEMENTS

disagreements = (
    df_evaluable[
        df_evaluable[
            'Researcher_Consensus'
        ]
        !=
        df_evaluable[
            'Behavioral_Profile'
        ]
    ]
)


print(
    f"\nModel / researcher disagreements: "
    f"{len(disagreements)}"
)

display(
    disagreements[
        [
            RESPONDENT_ID_COLUMN,
            'Q7_Risk_Preference',
            'Q9_Loss_Reaction',
            'Researcher_Consensus',
            'Behavioral_Profile'
        ]
    ]
)


This code cell analyzes the completed researcher-consensus annotation CSV to validate the XLM-RoBERTa behavioral classification. It loads the annotated data, cleans the researcher labels, and determines a majority consensus label for each respondent. It then merges this consensus with the model's predictions, calculates various classification metrics (accuracy, precision, recall, F1-score), generates a classification report and confusion matrix, and displays any disagreements between the model and the researcher consensus.

### 6.3.3 XLM-R VALIDATION SUMMARY


In [ ]:
# XLM-R VALIDATION SUMMARY

classification_report_dict = (
    classification_report(
        y_true,
        y_pred,
        labels=VALID_BEHAVIOR_LABELS,
        zero_division=0,
        output_dict=True
    )
)


paper_xlmr_metrics = pd.DataFrame({

    'Metric': [
        'Evaluated Profiles',
        'Accuracy',
        'Macro Precision',
        'Macro Recall',
        'Macro F1',
        'Weighted F1'
    ],

    'Value': [
        len(df_evaluable),
        accuracy,
        macro_precision,
        macro_recall,
        macro_f1,
        weighted_f1
    ]
})


print(
    "\n=== XLM-R VALIDATION SUMMARY ==="
)

display(
    paper_xlmr_metrics
)


per_class_records = []


for class_label in VALID_BEHAVIOR_LABELS:

    class_metrics = (
        classification_report_dict[
            class_label
        ]
    )

    per_class_records.append({

        'Behavioral_Profile':
            class_label,

        'Precision':
            class_metrics[
                'precision'
            ],

        'Recall':
            class_metrics[
                'recall'
            ],

        'F1_Score':
            class_metrics[
                'f1-score'
            ],

        'Support':
            class_metrics[
                'support'
            ]
    })


paper_xlmr_per_class = (
    pd.DataFrame(
        per_class_records
    )
)


print(
    "\n=== XLM-R PER-CLASS RESULTS ==="
)

display(
    paper_xlmr_per_class
)


print(
    "\n=== MODEL / RESEARCHER DISAGREEMENTS ==="
)

display(
    disagreements[
        [
            RESPONDENT_ID_COLUMN,
            'Q7_Risk_Preference',
            'Q9_Loss_Reaction',
            'Researcher_Consensus',
            'Behavioral_Profile'
        ]
    ]
)


paper_xlmr_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_xlmr_validation_metrics.csv'
    ),
    index=False
)


paper_xlmr_per_class.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_xlmr_per_class_metrics.csv'
    ),
    index=False
)


This code cell summarizes the XLM-RoBERTa validation results. It compiles overall metrics such as accuracy, macro F1, and weighted F1 into a DataFrame. It also presents per-class metrics (precision, recall, F1-score, support) for 'Low Risk Tolerance', 'Neutral', and 'High Risk Tolerance'. Finally, it reiterates and displays the disagreements between the model and the researcher consensus, saving all these metrics to CSV files.

## 6.4 FINAL PIPELINE DISTRIBUTION ANALYSIS


In [ ]:
def create_distribution_table(
    dataframe,
    column,
    order=None
):

    counts = (
        dataframe[
            column
        ]
        .value_counts()
    )


    percentages = (
        dataframe[
            column
        ]
        .value_counts(
            normalize=True
        )
        .mul(100)
    )


    result = pd.DataFrame({

        'Count':
            counts,

        'Percentage':
            percentages
    })


    if order is not None:

        result = (
            result.reindex(
                order
            )
            .fillna(0)
        )


    return (
        result.round(2)
    )


# FINANCIAL CAPACITY

paper_final_capacity_distribution = (
    create_distribution_table(
        df_compiled,
        'Financial_Capacity',
        CAPACITY_ORDER
    )
)


# BEHAVIORAL PROFILE

paper_final_behavior_distribution = (
    create_distribution_table(
        df_compiled,
        'Behavioral_Profile',
        BEHAVIOR_ORDER
    )
)


# INVESTMENT ASSESSMENT

paper_final_assessment_distribution = (
    create_distribution_table(
        df_compiled,
        'Investment_Assessment'
    )
)


# RULE RESULT

paper_final_rule_distribution = (
    create_distribution_table(
        df_compiled,
        'Rule_Result'
    )
)


print(
    "\n=== FINAL FINANCIAL CAPACITY DISTRIBUTION ==="
)

display(
    paper_final_capacity_distribution
)


print(
    "\n=== FINAL BEHAVIORAL PROFILE DISTRIBUTION ==="
)

display(
    paper_final_behavior_distribution
)


print(
    "\n=== FINAL INVESTMENT ASSESSMENT DISTRIBUTION ==="
)

display(
    paper_final_assessment_distribution
)


print(
    "\n=== FINAL RULE-RESULT DISTRIBUTION ==="
)

display(
    paper_final_rule_distribution
)


print(
    "\n=== FEASIBILITY SCORE SUMMARY ==="
)

display(
    df_compiled[
        'Feasibility_Score'
    ]
    .describe()
    .to_frame(
        name='Feasibility_Score'
    )
)


print(
    "\nMarket Signal Used for This Pipeline Run:",
    final_market_signal
)


# FINAL ASSESSMENT FIGURE

plt.figure(
    figsize=(10, 5)
)

paper_final_assessment_distribution[
    'Count'
].plot(
    kind='bar'
)

plt.title(
    'Final Investment Assessment Distribution'
)

plt.xlabel(
    'Investment Assessment'
)

plt.ylabel(
    'Number of Respondents'
)

plt.xticks(
    rotation=25,
    ha='right'
)

plt.tight_layout()


plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        'figure_final_assessment_distribution.png'
    ),
    dpi=300,
    bbox_inches='tight'
)


plt.show()


This code cell performs a final pipeline distribution analysis. It creates distribution tables for Financial Capacity, Behavioral Profile, Investment Assessment, and Rule Result from the `df_compiled` DataFrame, showing counts and percentages. It also displays a summary of the `Feasibility_Score` and the market signal used. A bar plot illustrating the final investment assessment distribution is generated and saved.

## 6.5 Qualitative Recommendation Evaluation

### 6.5.1 QUALITATIVE EVALUATION PACKET (FORMAT)


In [ ]:
# 6.5.1 CREATE QUALITATIVE EVALUATION PACKET

QUALITATIVE_SAMPLE_SIZE = min(
    15,
    len(df_compiled)
)


qualitative_sample = (
    df_compiled[
        [
            RESPONDENT_ID_COLUMN,
            'Financial_Capacity',
            'Behavioral_Profile',
            'Market_Signal',
            'Feasibility_Score',
            'System_Rule',
            'Investment_Assessment',
            'Recommendation_Advice'
        ]
    ]
    .sample(
        n=QUALITATIVE_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
    .reset_index(
        drop=True
    )
)


qualitative_sample.insert(
    0,
    'Evaluation_ID',
    np.arange(
        1,
        len(qualitative_sample) + 1
    )
)


# Blank 1–5 Likert ratings
qualitative_sample[
    'Clarity_Rating'
] = ""

qualitative_sample[
    'Transparency_Rating'
] = ""

qualitative_sample[
    'Relevance_Rating'
] = ""

qualitative_sample[
    'Perceived_Usefulness_Rating'
] = ""

qualitative_sample[
    'Optional_Comments'
] = ""


print(
    "\n=== QUALITATIVE RECOMMENDATION "
    "EVALUATION PACKET ==="
)


display(
    qualitative_sample
)


qualitative_filename = os.path.join(
    OUTPUT_DIR,
    'qualitative_recommendation_evaluation.csv'
)


qualitative_sample.to_csv(
    qualitative_filename,
    index=False
)


files.download(
    qualitative_filename
)


This code cell creates a qualitative evaluation packet for recommendation assessment. It samples a small subset of respondents from `df_compiled`, including their financial capacity, behavioral profile, market signal, feasibility score, system rule, investment assessment, and recommendation advice. Blank columns are added for Likert scale ratings (Clarity, Transparency, Relevance, Perceived Usefulness) and optional comments, and this packet is saved as a CSV for external qualitative evaluation.

### 6.5.2 COMPLETED QUALITATIVE EVALUATION


In [ ]:
# ANALYZE COMPLETED QUALITATIVE EVALUATION

print(
    "Upload the completed qualitative evaluation CSV:"
)


uploaded_qualitative = (
    files.upload()
)


qualitative_result_filename = (
    next(
        iter(
            uploaded_qualitative
        )
    )
)


df_qualitative_results = pd.read_csv(
    io.BytesIO(
        uploaded_qualitative[
            qualitative_result_filename
        ]
    )
)


rating_columns = [
    'Clarity_Rating',
    'Transparency_Rating',
    'Relevance_Rating',
    'Perceived_Usefulness_Rating'
]


for column in rating_columns:

    df_qualitative_results[
        column
    ] = pd.to_numeric(
        df_qualitative_results[
            column
        ],
        errors='coerce'
    )


    invalid_ratings = (
        ~df_qualitative_results[
            column
        ]
        .between(
            1,
            5
        )
        &
        df_qualitative_results[
            column
        ]
        .notna()
    )


    if invalid_ratings.any():

        raise ValueError(
            f"Invalid Likert rating found "
            f"in {column}. Ratings must be 1–5."
        )


qualitative_summary_records = []


for column in rating_columns:

    values = (
        df_qualitative_results[
            column
        ]
        .dropna()
    )


    qualitative_summary_records.append({

        'Dimension':
            column.replace(
                '_Rating',
                ''
            )
            .replace(
                '_',
                ' '
            ),

        'N':
            len(values),

        'Mean':
            values.mean(),

        'Median':
            values.median(),

        'Std_Dev':
            values.std()
    })


paper_qualitative_summary = (
    pd.DataFrame(
        qualitative_summary_records
    )
    .round(2)
)


print(
    "\n=== QUALITATIVE EVALUATION SUMMARY ==="
)


display(
    paper_qualitative_summary
)


# Rating frequency table
rating_frequency_tables = {}


for column in rating_columns:

    frequency = (
        df_qualitative_results[
            column
        ]
        .value_counts()
        .sort_index()
        .reindex(
            [1, 2, 3, 4, 5],
            fill_value=0
        )
    )

    rating_frequency_tables[
        column
    ] = frequency


paper_qualitative_frequencies = (
    pd.DataFrame(
        rating_frequency_tables
    )
)


print(
    "\n=== LIKERT RESPONSE FREQUENCIES ==="
)


display(
    paper_qualitative_frequencies
)


paper_qualitative_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_qualitative_summary.csv'
    ),
    index=False
)


paper_qualitative_frequencies.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_qualitative_frequencies.csv'
    )
)


This code cell analyzes the completed qualitative evaluation data. It prompts the user to upload the CSV containing the qualitative ratings, then reads the file into a DataFrame. It converts the Likert scale ratings to numeric format and validates them to ensure they fall within the 1-5 range. Finally, it calculates and displays summary statistics (mean, median, std dev) for each rating dimension and shows the frequency distribution of responses, saving these summaries to CSV files.

## 6.6 RESULTS SUMMARY


In [ ]:
# 6.6 MASTER RESULTS SUMMARY

master_result_records = []


# PIPELINE

master_result_records.extend([

    {
        'Module':
            'Dataset',

        'Metric':
            'Final Integrated Respondents',

        'Value':
            len(df_compiled)
    },

    {
        'Module':
            'Market',

        'Metric':
            'Current Market Signal',

        'Value':
            final_market_signal
    }
])


# FCM

master_result_records.extend([

    {
        'Module':
            'Fuzzy C-Means',

        'Metric':
            'FPC',

        'Value':
            round(
                fpc,
                4
            )
    },

    {
        'Module':
            'Fuzzy C-Means',

        'Metric':
            'Silhouette Score',

        'Value':
            round(
                sil_score,
                4
            )
    }
])


# PROPHET

if 'avg_rmse' in globals():

    master_result_records.extend([

        {
            'Module':
                'Prophet',

            'Metric':
                'RMSE',

            'Value':
                round(
                    avg_rmse,
                    2
                )
        },

        {
            'Module':
                'Prophet',

            'Metric':
                'MAE',

            'Value':
                round(
                    avg_mae,
                    2
                )
        },

        {
            'Module':
                'Prophet',

            'Metric':
                'MAPE (%)',

            'Value':
                (
                    round(
                        avg_mape,
                        2
                    )
                    if not np.isnan(
                        avg_mape
                    )
                    else np.nan
                )
        }
    ])


# XLM-R — ESEARCHER VALIDATION

if 'accuracy' in globals():

    master_result_records.extend([

        {
            'Module':
                'XLM-RoBERTa',

            'Metric':
                'Accuracy',

            'Value':
                round(
                    accuracy,
                    4
                )
        },

        {
            'Module':
                'XLM-RoBERTa',

            'Metric':
                'Macro F1',

            'Value':
                round(
                    macro_f1,
                    4
                )
        },

        {
            'Module':
                'XLM-RoBERTa',

            'Metric':
                'Weighted F1',

            'Value':
                round(
                    weighted_f1,
                    4
                )
        }
    ])


paper_master_results = (
    pd.DataFrame(
        master_result_records
    )
)


print(
    "\n=== MASTER RESULTS SUMMARY ==="
)


display(
    paper_master_results
)


paper_master_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'paper_master_results_summary.csv'
    ),
    index=False
)


This code cell compiles a master results summary, bringing together key metrics and findings from various modules of the financial advisory prototype. It includes the number of final integrated respondents, the current market signal, FPC and Silhouette Score for FCM, RMSE, MAE, and MAPE for Prophet, and accuracy, Macro F1, and Weighted F1 for XLM-RoBERTa. This consolidated summary is displayed and saved to a CSV file.

# 7.0 DATA EXPORTING


In [ ]:
final_columns = [

    RESPONDENT_ID_COLUMN,

    'Financial_Capacity',

    'Membership_Low',
    'Membership_Medium',
    'Membership_High',

    'Behavioral_Profile',
    'Behavioral_Confidence',

    'Q7_Risk_Profile',
    'Q9_Loss_Profile',

    'Market_Signal',

    'Rule_Delta',
    'Feasibility_Score',

    'System_Rule',
    'Rule_Result',
    'System_Reasoning',

    'Investment_Assessment',
    'Recommendation_Advice'
]


df_final_output = (
    df_compiled[
        final_columns
    ]
    .copy()
)


print(
    "\n=== FINAL ADVISORY OUTPUT ==="
)


display(
    df_final_output
)


output_filename = os.path.join(
    OUTPUT_DIR,
    'final_advisory_results.csv'
)


df_final_output.to_csv(
    output_filename,
    index=False
)


print(
    f"\nFinal advisory results exported to: "
    f"{output_filename}"
)


files.download(
    output_filename
)


This code cell prepares and exports the final advisory results. It selects a specific set of columns from `df_compiled`, including financial capacity, behavioral profile, market signal, feasibility scores, system rules, and recommendation advice, to create `df_final_output`. This DataFrame, representing the complete output of the advisory system, is displayed, saved to a CSV file, and provided as a download link.
